In [ ]:
from cits.methods import cits_full_weighted #RUN
import multiprocessing
import numpy as np
import time
import os
import pickle as pkl
import matplotlib.pyplot as plt

In [ ]:
# %% RUN
def smooth_spikes( P, kernel_width , bin_size ):
    # Rahul
    """
    P: #trials x #units x #time-points sized ndarray
    kernel_param: width of kernel (in samplingrate*1000 units)

    """

    import time
    import scipy.stats, scipy.signal
    import scipy as sp

    start = time.time() # tic Measures function running speed. Works only on Macs

    #trials = P.shape[0]
    #units = P.shape[1]
    #bins = P.shape[2]

    # Create fake data for testing purposes
    #    array = np.random.randint(0,2,(trials,bins))
    #    sigma = 45

    #Define kernel
    kernel_param = kernel_width/bin_size
    sigma = kernel_param/6.
    edges = np.arange(-3*sigma,3*sigma+1,1)
    kernel = sp.stats.norm.pdf(edges,0, sigma) #Use a gaussian function
    kernel = kernel/sum(kernel)
    
    #Compute Spike Density Function for all trials
    #X = np.zeros((trials,units,bins))
    X = sp.signal.convolve(P, kernel[None,:,None], mode='same',method='direct')

    print('Run time for SDF function was ' + str(round(time.time()-start)) + 'seconds') # toc
    return X

In [ ]:
#%% RUN
import os
import json
import shutil
import time
import numpy as np
import pandas as pd
from scipy.special.orthogonal import chebyc
from tqdm import tqdm
import pickle as pkl
import matplotlib
import matplotlib.pyplot as plt
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
#%%
data_directory = 'data/' # must be updated to a valid directory in your filesystem

manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
sessions = cache.get_session_table()
probes = cache.get_probes()
channels = cache.get_channels()
units = cache.get_units()
#%%
sess_id = 791319847
#%%
data = cache.get_session_data(sess_id)
#%%
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)

#%%
stim_label='drifting_gratings_contrast'
allencon = A_Config(data_directory,sess_id,stim_label)
stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
bin_size=0.01
kernel_width=0.1
# X={}
# P={}
# for i, stim_label in enumerate(stim_label_list):
#     print(i)
#     allencon = A_Config(data_directory,sess_id,stim_label)
#     P[stim_label] = pkl.load(open(allencon.name+'_bin_{}_P.p'.format(bin_size),'rb'))
#     X[stim_label] = smooth_spikes(P[stim_label].values,kernel_width,bin_size)


In [ ]:
import os
import pickle as pkl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

stimuli_trials = {
    'natural_scenes': [54],
    'static_gratings': [2],
    'gabors': [1],
    'flashes': [14]
}

methods_map = {
    'gc1': 'gc1',
    'gc2': 'gc2',
    'pc': 'pc',
    'tpc': 'tpc',
    'cits': 'tsci'  # method_name in saved results
}

alpha = 0.05
bin_size = 0.01
niter = 50
session_id = 791319847
results_dir = 'save/'
afc_dir = 'graphs/'  # directory where AFC edge lists are saved

def matrix_to_edges(adj):
    # Returns a set of directed (src, dst) edges from a binary matrix
    return set(zip(*np.where(np.array(adj) > 0)))

results = []

for stim, trial_list in stimuli_trials.items():
    for idx in trial_list:
        # Load AFC
        afc_path = os.path.join(afc_dir, f'ID{session_id}_{stim}_trial{idx}_afc_edges.pkl')
        try:
            afc_edges = set(pkl.load(open(afc_path, 'rb')))
        except:
            print(f"Missing AFC for {stim} trial {idx}")
            continue

        for label, method in methods_map.items():
            file = f"{results_dir}/ID{session_id}_{stim}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
            if not os.path.exists(file):
                print(f"Missing {method} result for {stim} trial {idx}")
                continue

            try:
                result = pkl.load(open(file, 'rb'))
                adj_matrix = result['adj']
                cfc_edges = matrix_to_edges(adj_matrix)
            except:
                print(f"Error reading file: {file}")
                continue

            only_afc = afc_edges - cfc_edges
            only_cfc = cfc_edges - afc_edges
            both = afc_edges & cfc_edges

            results.append({
                'stimulus': stim,
                'trial': idx,
                'method': label,
                'only_afc': len(only_afc),
                'only_cfc': len(only_cfc),
                'both': len(both),
                'afc_total': len(afc_edges),
                'cfc_total': len(cfc_edges)
            })

# Convert to DataFrame
df = pd.DataFrame(results)
from ace_tools import display_dataframe_to_user
display_dataframe_to_user("CFC_AFC_MatrixComparison", df)

# Plot per stimulus
for (stim, trial), group in df.groupby(['stimulus', 'trial']):
    fig, ax = plt.subplots(figsize=(8, 5))
    bar_width = 0.25
    x = np.arange(len(group))

    ax.bar(x - bar_width, group['only_afc'], width=bar_width, label='AFC Only')
    ax.bar(x, group['both'], width=bar_width, label='Both')
    ax.bar(x + bar_width, group['only_cfc'], width=bar_width, label='CFC Only')

    ax.set_xticks(x)
    ax.set_xticklabels(group['method'])
    ax.set_ylabel("Edge Count")
    ax.set_title(f'Edge Overlap for {stim}, Trial {trial}')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
#SKIP
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_ = np.asarray(np.mean(np.sum(P[stim_label]>0, axis=1),axis=0)>(2*P[stim_label].shape[1]/10))
    print(np.sum(units_to_use_idx_))
    pkl.dump(units_to_use_idx_,open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'wb'))

#%%
units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

#%%
for stim_label in stim_label_list[:-1]:
    for idx in range(60):
        allencon = A_Config(data_directory,sess_id,stim_label)
        mul = 50
        idx1 = mul*idx
        X_out = X[stim_label][idx1,:,:]
        for idx2 in range(idx1+1,idx1+mul):
            X_out = np.row_stack((X_out,X[stim_label][idx2,:,:]))
        pkl.dump(X_out,open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'wb'))
for stim_label in stim_label_list[3:]:
    for idx in range(3):
        mul=50
        idx1 = mul*idx
        allencon = A_Config(data_directory,sess_id,stim_label)
        X_out = X[stim_label][idx1,:,:]
        for idx2 in range(idx1+1,idx1+mul):
            X_out = np.row_stack((X_out,X[stim_label][idx2,:,:]))
        pkl.dump(X_out,open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'wb'))

In [ ]:
alpha = 0.05#float(sys.argv[1])

thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01] #float(sys.argv[2])

for idx0 in range(4):

    stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
    stim_label = stim_label_list[idx0]

    if idx0==0:
        idxlist=range(60)#[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]
    elif idx0==2:
        idxlist=range(60)#[2,3]#[0,2,3,4,5,6,7,8,9]
    elif idx0==3:
        idxlist=range(60)#[1]#[0,1,2]
    else:
        idxlist=range(3)#[14]#[10,11,12,13,14,15,16,17,18,19]
#idx = 10
#idx = 0
    for idx in idxlist:
        sess_id = 791319847
        niter = 50
        method_name = 'tsci'#'tsci'#'gc'#'gc'#'neuropc'
        data_dir =  'data/'# 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\data'#'/homes/rbiswas1/babaice_data/''C:\\Users\\BN Laptop\\OneDrive - UW\\research\\projects_git\\mice-aibs\\data'#
        results_dir =  'save/' #'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#/homes/rbiswas1/babaice_save2/''C:\\Users\\BN Laptop\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'#
        #explist = {0:['N-5_T-300'],1:['N-5_T-300_noise-small','N-5_T-300_noise-medium','N-5_T-300_noise-strong'],2:['N-5_T-300_weak','N-5_T-300_medium','N-5_T-300_strong'],3:['N-5_T-300_weak','N-5_T-300_medium','N-5_T-300_strong'],4:['N-5_T-300'],5:['N-5_T-300'],6:['N-12_T-4600'],7:['N-5_T-2000'],8:['N-5_T-250'],9:['N-5_T-2000'],10:['N-5_T-250']}
        bin_size=0.01
        kernel_width=0.1
        allencon = A_Config(data_dir,sess_id,stim_label)
        allencon_save = A_Config(results_dir,sess_id,stim_label)
        subsampsize = 250

        results={}
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)


        name = allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx)
        units2use = pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb'))
        data = pkl.load(open(name,'rb'))
        print(data.shape)
        data = data[:,units2use]
        data -= data.mean(axis=0)
        data /= data.std(axis=0)
        data1 = data.T#np.row_stack((data,data+2*np.mean(data),data+3*np.mean(data),data+4*np.mean(data)))
        start_time = time.time()
        lag = 1
        print("shape of data "+str(data1.shape))
        print("T: "+str(data.shape[0])+' duplexed: '+str(data1.shape[0]))
        if method_name == 'neuropc':
            val_matrix_threshlist, causaleff, causaleff2 = val_neuropc(data1,lag,subsampsize=subsampsize,n_iter=niter,alpha=alpha,thresh_list=thresh_list)
        elif method_name == 'gc':
            val_matrix_threshlist, causaleff, causaleff2 = val_gc(data1,lag,alpha)
        elif method_name == 'tsci':
            val_matrix_threshlist, causaleff = cits_full_weighted(data1,lag,alpha, cond_dep = 'cond_dep_pcorr', thresh = 100)
            #causaleff = val_matrix_threshlist
            causaleff2 = causaleff
        runtimes=time.time() - start_time
        print(runtimes)
        results['runtimes'] = runtimes#np.array(runtimes).tolist()
        results['adj']=val_matrix_threshlist#outadj[thresh]#np.array(outadj).tolist()
        results['ce']=causaleff#outce[thresh]#np.array(outce).tolist()
        results['ce2']=causaleff2
        results['pvals']=1-val_matrix_threshlist#pvals[thresh]
        # Save data
        with open(results_file, 'wb') as file1:
            pkl.dump(results, file1)

In [ ]:
import os
import pickle as pkl
import time
import multiprocessing
from statsmodels.tsa.stattools import adfuller

class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus
        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)

alpha = 0.05
thresh_list = [0.75, 0.70, 0.65, 0.60, 0.55, 0.50, 0.45, 0.40,
               0.35, 0.30, 0.25, 0.20, 0.15, 0.10, 0.05, 0.01]

def test_stationarity_adf(timeseries, alpha=0.05):
    try:
        result = adfuller(timeseries)
        return result[1]  # p-value
    except:
        return None

def calc_stationarity(idx):
    stim_label = stim_label_list[idx0]
    sess_id = 791319847
    data_dir = 'data/'
    bin_size = 0.01

    allencon = A_Config(data_dir, sess_id, stim_label)
    name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size, idx)

    units2use = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label), 'rb'))
    data = pkl.load(open(name, 'rb'))  # shape (T, N)

    print(f"\n>> Loaded trial {idx} of stimulus '{stim_label}' with shape {data.shape}")
    data = data[:, units2use]
    data -= data.mean(axis=0)
    data /= data.std(axis=0)

    print(f"\n--- ADF Stationarity test for trial {idx} ({stim_label}) ---")
    pvals = []
    for i in range(data.shape[1]):
        pval = test_stationarity_adf(data[:, i])
        pvals.append(pval)
        if pval is not None:
            status = "Stationary" if pval < alpha else "Non-stationary"
            print(f"Neuron {i}: p = {pval:.4f} → {status}")
        else:
            print(f"Neuron {i}: ADF test failed or invalid input.")

    n_valid = len([p for p in pvals if p is not None])
    n_stationary = len([p for p in pvals if p is not None and p < alpha])
    print(f"\nSummary: {n_stationary}/{n_valid} neurons are stationary in trial {idx}\n")


# Loop through all stimuli
for idx0 in range(4):
    stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
    stim_label = stim_label_list[idx0]
    if idx0 == 3:
        idxlist = range(3)
    else:
        idxlist = range(60)

    print(f"\n=== Processing Stimulus: {stim_label} ===")
    for idx1 in idxlist:
        calc_stationarity(idx1)

In [ ]:
non_stationary_summary = []
total_trials = 0
trials_with_nonstationary = 0
total_nonstationary_neurons = 0
total_neurons_all_trials = 0

stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
for idx0, stim_label in enumerate(stim_label_list):
    if idx0 == 3:
        idxlist = range(3)
    else:
        idxlist = range(60)

    print(f"\n=== Processing Stimulus: {stim_label} ===")
    for idx1 in idxlist:
        sess_id = 791319847
        data_dir = 'data/'
        bin_size = 0.01

        allencon = A_Config(data_dir, sess_id, stim_label)
        name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size, idx1)

        try:
            units2use = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label), 'rb'))
            data = pkl.load(open(name, 'rb'))  # shape (T, N)
        except FileNotFoundError:
            print(f"File missing for trial {idx1}, skipping.")
            continue

        total_trials += 1
        data = data[:, units2use]
        data -= data.mean(axis=0)
        data /= data.std(axis=0)

        pvals = [test_stationarity_adf(data[:, i]) for i in range(data.shape[1])]
        valid_neurons = [p for p in pvals if p is not None]
        non_stationary = [p for p in valid_neurons if p >= alpha]

        n_valid = len(valid_neurons)
        n_non_stationary = len(non_stationary)

        if n_non_stationary > 0:
            trials_with_nonstationary += 1
        total_nonstationary_neurons += n_non_stationary
        total_neurons_all_trials += n_valid

        non_stationary_summary.append({
            "stimulus": stim_label,
            "trial": idx1,
            "non_stationary_count": n_non_stationary,
            "total_neurons": n_valid
        })

# Final Summary
print("\n================ SUMMARY ================")
print(f"Total trials processed: {total_trials}")
print(f"Trials with ≥1 non-stationary neuron: {trials_with_nonstationary}")
print(f"Total neurons tested (valid ADF): {total_neurons_all_trials}")
print(f"Total non-stationary neurons: {total_nonstationary_neurons}")

if total_trials > 0:
    pct_trials_nonstationary = 100 * trials_with_nonstationary / total_trials
    print(f"→ {pct_trials_nonstationary:.2f}% of trials had ≥1 non-stationary neuron")

if total_neurons_all_trials > 0:
    pct_neurons_nonstationary = 100 * total_nonstationary_neurons / total_neurons_all_trials
    print(f"→ {pct_neurons_nonstationary:.2f}% of all neurons were non-stationary")
print("=========================================")


In [ ]:
#%% RUN
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
data_directory = 'data/'
manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
stim_label=stim_label_list[0]
sess_id = 791319847
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)
allencon = A_Config(data_directory,sess_id,stim_label)
#X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
#X_load.shape
#%% units labeled
data = cache.get_session_data(sess_id)

units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]
#%%
def plot_matrix(mat,vmnmx,id,allencon,namedetails='',method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    #m = mat - np.diag(np.diag(mat))
    m = mat
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    # if method == 'neuropc':
    #     vmnmx = 1#3
    # elif method == 'glm':
    #     vmnmx = 0.0008#0.001
    # elif method == 'granger':
    #     vmnmx = 6
    norm = mpl.colors.Normalize(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))##
    img=ax.matshow(m,cmap='RdBu_r',norm=norm)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(allencon.name+id+'_'+method+'_'+namedetails+'.png',format='png',dpi=600)
def plot_matrix_bw(mat,vmnmx,id,allencon,namedetails='',method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    #m = mat - np.diag(np.diag(mat))
    m = mat
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    # if method == 'neuropc':
    #     vmnmx = 1#3
    # elif method == 'glm':
    #     vmnmx = 0.0008#0.001
    # elif method == 'granger':
    #     vmnmx = 6
    #norm = mpl.colors.Normalize(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))##
    cmap = mpl.colors.ListedColormap(['white', 'black'])
    img=ax.matshow(m!=0,cmap=cmap)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(allencon.name+id+'_bw_'+method+'_'+namedetails+'.png',format='png',dpi=600)



In [ ]:
unit_labels_permuted.shape

In [ ]:
#%%
from neuropixels_testresults import convert_fcmat2units
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)
        
sess_id = 791319847
save_directory = 'save/'

stim_label = 'natural_scenes'

bin_size = 0.01 
method_name = 'tsci'
niter = 50
allencon_save = A_Config(save_directory,sess_id,stim_label)

idx = 0
alpha = 0.05

result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(0.01,alpha,idx,niter),'rb'))
ce = result['ce']
adj = result['adj']

ce2 = convert_fcmat2units(ce-np.diag(np.diag(ce)), stim_label, units,permute)
plt.matshow(ce2)
plt.colorbar()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle as pkl
from collections import defaultdict

# Load result
result = pkl.load(open(
    allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(0.01, alpha, idx, niter), 'rb'))

ce = result['ce']
adj = result['adj']

# Assuming `units` and `permute` are defined somewhere in your code
ce2 = convert_fcmat2units(ce - np.diag(np.diag(ce)), stim_label, units, permute)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(ce2, vmin=-0.1, vmax=0.1, cmap='bwr')

# Add a smaller colorbar
cbar = fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04, extend = 'both')
cbar.set_label('Connection strength')

# Group labels
labels = unit_labels_permuted
label_pos = {}
label_count = defaultdict(int)

# Count and store index positions of labels
for i, label in enumerate(labels):
    label_count[label] += 1

pos = 0
for label, count in label_count.items():
    label_pos[label] = pos + count // 2
    pos += count

# Set custom ticks
tick_pos = list(label_pos.values())
tick_labels = list(label_pos.keys())

ax.set_xticks(tick_pos)
ax.set_yticks(tick_pos)
ax.set_xticklabels(tick_labels, rotation=90)
ax.set_yticklabels(tick_labels)

plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
from collections import defaultdict
from neuropixels_testresults import convert_fcmat2units

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
data_directory = 'data/'
manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
stim_label=stim_label_list[0]
allencon = A_Config(data_directory,sess_id,stim_label)
#X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
#X_load.shape
#%% units labeled
data = cache.get_session_data(sess_id)

units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]

save_directory = 'save/'
plot_save_directory = 'figs/'  # Output folder for plots
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50

for idx0, stim_label in enumerate(stim_label_list):
    ls = 60 if idx0 < 3 else 3
    for idx in range(ls):
        filename = f'{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p'
        if not os.path.exists(filename):
            print(f"Missing: {filename}")
            continue

        # Load file
        with open(filename, 'rb') as f:
            result = pkl.load(f)
        
        ce = result['ce']
        ce2 = convert_fcmat2units(ce - np.diag(np.diag(ce)), stim_label, units, permute)

        # Group labels for ticks
        labels = unit_labels_permuted
        label_pos = {}
        label_count = defaultdict(int)
        for i, label in enumerate(labels):
            label_count[label] += 1

        pos = 0
        for label, count in label_count.items():
            label_pos[label] = pos + count // 2
            pos += count

        tick_pos = list(label_pos.values())
        tick_labels = list(label_pos.keys())

        # Plot
        fig, ax = plt.subplots(figsize=(10, 8))
        cax = ax.matshow(ce2, vmin=-0.1, vmax=0.1, cmap='bwr')

        # Colorbar (smaller)
        cbar = fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Connection Strength')

        # Ticks
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_labels, rotation=90)
        ax.set_yticks(tick_pos)
        ax.set_yticklabels(tick_labels)

        # Save
        basename = f"{plot_save_directory}ID{sess_id}_{stim_label}_idx{idx}"
        fig.tight_layout()
        fig.savefig(basename + ".pdf", bbox_inches='tight')
        fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
        plt.close(fig)

        print(f"Saved: {basename}.pdf/.png")


In [ ]:
plot_save_directory = 'figs/'
save_directory = 'save/'
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle as pkl
import networkx as nx
from collections import defaultdict
from neuropixels_testresults import convert_fcmat2units

# Graph settings
os.makedirs(plot_save_directory, exist_ok=True)
unique_labels = list(np.unique(unit_labels_permuted))
color_map = dict(zip(unique_labels, plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))))

def get_grouped_positions(labels, spacing=5):
    pos = {}
    x_start = 0
    y_step = 1
    for area in labels_ordered:
        indices = [i for i, l in enumerate(labels) if l == area]
        for j, idx in enumerate(indices):
            pos[idx] = (x_start, -j * y_step)
        x_start += spacing
    return pos

for stim_label in stim_label_list:
    # Initialize edge accumulators
    n_nodes = len(units)
    edge_strength_raw = np.zeros((n_nodes, n_nodes))
    edge_presence_raw = np.zeros((n_nodes, n_nodes))
    n_trials = 0

    ls = 60 if stim_label != 'flashes' else 3
    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)
        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        ce2 = convert_fcmat2units(ce, stim_label, units, permute)
        
        edge_strength_raw += ce2
        edge_presence_raw += (ce2 != 0).astype(float)

    if n_trials == 0:
        print(f"No data found for {stim_label}, skipping...")
        continue

    # Average over trials
    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials

    # Reorder using permute
    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]

    # Build graph
    G = nx.DiGraph()
    for i in range(n_nodes):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(n_nodes):
        for j in range(n_nodes):
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    # Plot graph
    fig, ax = plt.subplots(figsize=(14, 12))
    node_colors = [color_map[G.nodes[n]['area']] for n in G.nodes]
    pos = get_grouped_positions(unit_labels_permuted)

    for (u, v, d) in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=np.abs(d['weight']) * 5,
            alpha=d['stability'],
            edge_color='gray',
            arrows=True,
            ax=ax
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    # Legend
    for area, color in color_map.items():
        ax.plot([], [], color=color, marker='o', linestyle='None', label=area)
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), title='Brain Area')

    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials')
    ax.axis('off')

    # Save
    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.tight_layout()
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"Graph saved: {basename}.pdf/.png")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
from collections import defaultdict

# Ensure save path
os.makedirs(plot_save_directory, exist_ok=True)

# Define colors
unique_labels = list(np.unique(unit_labels_permuted))
color_map = dict(zip(unique_labels, plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))))

# Node grouping function
def get_grouped_positions(labels, spacing=5):
    pos = {}
    x_start = 0
    y_step = 1
    for area in labels_ordered:
        indices = [i for i, l in enumerate(labels) if l == area]
        for j, idx in enumerate(indices):
            pos[idx] = (x_start, -j * y_step)
        x_start += spacing
    return pos
def get_clustered_circle_positions(labels, min_spacing=8, base_cluster_radius=2):
    """
    Improved layout:
    - Regions arranged on a large circle
    - Each region's neuron cluster size depends on its size
    - Avoids overlapping between regions
    """
    pos = {}
    region_indices = defaultdict(list)

    # Group neuron indices by brain region
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_names = list(region_indices.keys())
    num_regions = len(region_names)
    region_sizes = {label: len(indices) for label, indices in region_indices.items()}

    # Estimate required space per region (based on neuron count)
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2

    # Assign regions evenly on circle, space depends on size
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)

        # Center of this cluster
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius

        # Place neurons in circular pattern
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos


def get_grouped_positions_scattered(labels, spacing=10, jitter=1.5):
    """Group nodes by labels with slight jittered circular layouts per group."""
    pos = {}
    group_centers = {}
    x_offset = 0

    for area in labels_ordered:
        indices = [i for i, l in enumerate(labels) if l == area]
        n = len(indices)
        if n == 0:
            continue

        # Set center of this group
        cx = x_offset
        cy = 0
        group_centers[area] = (cx, cy)

        # Arrange in a jittered circle
        theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for k, idx in enumerate(indices):
            dx = np.cos(theta[k]) * jitter
            dy = np.sin(theta[k]) * jitter
            pos[idx] = (cx + dx, cy + dy)

        # Update x offset for next group
        x_offset += spacing

    return pos

# Create index map from unit id → full unit array index
unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
print(f"Total units across stimuli: {len(units)}")

for stim_label in stim_label_list:
    print(f"\n--- Processing stimulus: {stim_label} ---")
    ls = 60 if stim_label != 'flashes' else 3

    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        print(f"Checking file: {file}")

        if not os.path.exists(file):
            print(f" --> File missing, skipping index {idx}")
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)
        
        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))  # remove self-loops
        print(f" --> Loaded CE matrix of shape {ce.shape} for idx {idx}")

        trial_units_idx = result.get('units', units_to_use_idx_[stim_label])
        trial_units = trial_units_idx  # FIXED: don't index again
        print(f" --> Trial uses {len(trial_units)} units")
        
        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError as e:
            print(f"ERROR: Unit ID {e} not found in full unit set")
            continue


        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += ce[i, j]
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1
        print(f" --> Aggregated index {idx} ({n_trials} trials accumulated)")

    if n_trials == 0:
        print(f"!!! No valid data found for stimulus {stim_label}, skipping graph")
        continue

    print(f"--> Total trials used for {stim_label}: {n_trials}")
    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    # Threshold for minimum stability (e.g., keep edges appearing in ≥ 30% of trials)
    min_stability = 0.30
    
    # Zero out unstable edges
    mask = stability < min_stability
    avg_strength[mask] = 0
    stability[mask] = 0

    # Reorder using permute
    print("Reordering matrix...")
    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]

    print("Building graph...")
    G = nx.DiGraph()
    for i in range(len(units)):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(len(units)):
        for j in range(len(units)):
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    print("Plotting graph...")
    fig, ax = plt.subplots(figsize=(14, 12))
    node_colors = [color_map[G.nodes[n]['area']] for n in G.nodes]
    #pos = get_grouped_positions(unit_labels_permuted)
    pos = get_clustered_circle_positions(unit_labels_permuted, min_spacing=12, base_cluster_radius=3.0)

    for (u, v, d) in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=np.abs(d['weight']) * 5,
            alpha=d['stability'],
            edge_color='gray',
            arrows=True,
            ax=ax
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    for area, color in color_map.items():
        ax.plot([], [], color=color, marker='o', linestyle='None', label=area)
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), title='Brain Area')

    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials')
    ax.axis('off')

    # Save
    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.tight_layout()
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved graph: {basename}.pdf and .png")


In [ ]:
trial_units_idx

In [ ]:
import os
import numpy as np
import pickle as pkl
from collections import defaultdict
from neuropixels_testresults import convert_fcmat2units


from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
data_directory = 'data/'
manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
stim_label=stim_label_list[0]
allencon = A_Config(data_directory,sess_id,stim_label)
#X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
#X_load.shape
#%% units labeled
data = cache.get_session_data(sess_id)

units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]

save_directory = 'save/'
output_directory = 'region_cfc/'  # Where to save compressed region-level matrices
os.makedirs(output_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50

# Step 1: Map region names to indices
region_to_indices = defaultdict(list)
for idx, region in enumerate(unit_labels_permuted):
    region_to_indices[region].append(idx)

regions = list(region_to_indices.keys())

# Step 2: Process each CFC
for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    for idx in range(ls):
        filename = f'{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p'
        if not os.path.exists(filename):
            print(f"Skipping: {filename}")
            continue

        with open(filename, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce2 = convert_fcmat2units(ce - np.diag(np.diag(ce)), stim_label, units, permute)
        abs_ce = np.abs(ce2)

        # Step 3: Create region-to-region compressed CFC (mean of non-zero absolute values)
        region_cfc = np.zeros((len(regions), len(regions)))
        for i, region_i in enumerate(regions):
            for j, region_j in enumerate(regions):
                indices_i = region_to_indices[region_i]
                indices_j = region_to_indices[region_j]
        
                submatrix = abs_ce[np.ix_(indices_i, indices_j)]
                nonzero_values = submatrix[submatrix != 0]
        
                if nonzero_values.size > 0:
                    region_cfc[i, j] = nonzero_values.mean()
                else:
                    region_cfc[i, j] = 0.0  # or np.nan if preferred

        # Step 4: Save compressed matrix
        output_file = f"{output_directory}/ID{sess_id}_{stim_label}_idx{idx}_region_cfc.npy"
        np.save(output_file, region_cfc)

        # Optional: save region labels once
        with open(f"{output_directory}/region_labels.npy", 'wb') as f:
            np.save(f, np.array(regions))

        print(f"Saved region CFC: {output_file}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# === Setup ===
region_cfc_dir = 'region_cfc/'
output_dir = 'representative_cfcs/'
os.makedirs(output_dir, exist_ok=True)

stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
sess_id = 791319847
threshold_ratio = 0.3  # 20%
percentage_max_display = 100  # for colorbar

# Load region labels
region_labels = np.load(os.path.join(region_cfc_dir, 'region_labels.npy'))
n_regions = len(region_labels)

for stim_label in stim_label_list:
    region_cfcs = []
    
    # Collect all files for the stimulus
    for fname in os.listdir(region_cfc_dir):
        if fname.startswith(f'ID{sess_id}_{stim_label}') and fname.endswith('_region_cfc.npy'):
            region_cfc = np.load(os.path.join(region_cfc_dir, fname))
            region_cfcs.append(region_cfc)
    
    if len(region_cfcs) == 0:
        print(f"No region CFCs found for {stim_label}")
        continue
    
    region_cfcs = np.array(region_cfcs)  # shape: [n_samples, n_regions, n_regions]
    n_samples = region_cfcs.shape[0]

    # === Calculate percentage and representative matrices ===
    #non_zero_counts = (region_cfcs != 0).sum(axis=0)  # shape: (n_regions, n_regions)
    #percentage_matrix = (non_zero_counts / n_samples) * 100
    #representative_matrix = (percentage_matrix >= (threshold_ratio * 100)).astype(int)
    # Get average value only over samples where (a,b) connection is non-zero
    region_cfcs = np.array(region_cfcs)  # shape: (n_samples, n_regions, n_regions)
    
    # Binary mask of (a,b) having non-zero connection across samples
    non_zero_mask = (region_cfcs != 0)
    
    # Percentage matrix
    non_zero_counts = non_zero_mask.sum(axis=0)
    percentage_matrix = (non_zero_counts / region_cfcs.shape[0]) * 100
    
    # Representative weighted matrix: average only across non-zero samples
    weighted_matrix = np.zeros((n_regions, n_regions))
    for i in range(n_regions):
        for j in range(n_regions):
            valid_vals = region_cfcs[:, i, j][non_zero_mask[:, i, j]]
            if len(valid_vals) > 0 and (len(valid_vals) / region_cfcs.shape[0]) >= 0.2:
                weighted_matrix[i, j] = valid_vals.mean()
            else:
                weighted_matrix[i, j] = 0.0
    representative_matrix = weighted_matrix
    # === Save both matrices ===
    np.save(os.path.join(output_dir, f"{stim_label}_percentage_matrix.npy"), percentage_matrix)
    np.save(os.path.join(output_dir, f"{stim_label}_representative_matrix.npy"), representative_matrix)

    # === Plot percentage matrix ===
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    cax1 = ax1.matshow(percentage_matrix, cmap='viridis', vmin=0, vmax=percentage_max_display)
    fig1.colorbar(cax1, ax=ax1, label='Presence (%)')
    ax1.set_xticks(np.arange(n_regions))
    ax1.set_yticks(np.arange(n_regions))
    ax1.set_xticklabels(region_labels, rotation=90)
    ax1.set_yticklabels(region_labels)
    fig1.tight_layout()
    fig1.savefig(os.path.join(output_dir, f"{stim_label}_percentage_matrix.png"), dpi=600, bbox_inches='tight')
    fig1.savefig(os.path.join(output_dir, f"{stim_label}_percentage_matrix.pdf"), bbox_inches='tight')
    plt.close(fig1)

    # === Plot representative matrix ===
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    cax2 = ax2.matshow(representative_matrix, cmap='Blues',vmin=0, vmax=0.08)
    fig2.colorbar(cax2, ax=ax2, label='Representative (≥30%)')
    ax2.set_xticks(np.arange(n_regions))
    ax2.set_yticks(np.arange(n_regions))
    ax2.set_xticklabels(region_labels, rotation=90)
    ax2.set_yticklabels(region_labels)
    fig2.tight_layout()
    fig2.savefig(os.path.join(output_dir, f"{stim_label}_representative_matrix.png"), dpi=600, bbox_inches='tight')
    fig2.savefig(os.path.join(output_dir, f"{stim_label}_representative_matrix.pdf"), bbox_inches='tight')
    plt.close(fig2)

    print(f"Done: {stim_label} ✅")


In [ ]:
import os
import re
import pickle
from neuropixels_testresults import convert_fcmat2units

# Function to gather filenames for each stimulus type from a directory
def gather_files_from_directory(directory, bin_size, idx_range):
    stimulus_files = {}  # Dictionary to hold lists of files for each stimulus
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        # Check if the file matches the format 'name_bin_{bin_size}_X_idx-{idx}.p'
        match = re.match(r"(.+)_bin_(\d+)_X_idx-(\d+)\.p", filename)
        if match:
            # Extract the name (which includes the stimulus type), bin_size, and idx
            base_name = match.group(1)
            file_bin_size = int(match.group(2))
            file_idx = int(match.group(3))
            
            # Check if the bin_size and idx are within the expected range
            if file_bin_size == bin_size and file_idx in idx_range:
                # Extract stimulus type from base name (assuming format: 'save_dir/ID{sess_id}_stimulus')
                # For example: "save_dir/ID101_stimulusA" -> stimulusA
                stimulus = base_name.split('_')[-1]
                
                if stimulus not in stimulus_files:
                    stimulus_files[stimulus] = []
                
                stimulus_files[stimulus].append(filename)
    
    return stimulus_files

# Example usage
directory = "data/"  # Replace with your directory path
bin_size = 10  # The bin size to match
idx_range = range(60)  # Assuming indices 0-4 are valid

# Get all file names categorized by stimulus type
stimulus_files = gather_files_from_directory(directory, bin_size, idx_range)

# Example output
for stimulus, files in stimulus_files.items():
    print(f"Files for {stimulus}:")
    for file in files:
        print(file)


In [ ]:
#USE THIS FOR UNITS STUFF
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
from collections import defaultdict
import matplotlib.patches as mpatches

import os
import numpy as np
import pickle as pkl
from collections import defaultdict
from neuropixels_testresults import convert_fcmat2units

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
data_directory = 'data/'
manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
sess_id = 791319847

#stim_label=stim_label_list[0]
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)
#allencon = A_Config(data_directory,sess_id,stim_label)
#X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
#X_load.shape
#%% units labeled
data = cache.get_session_data(sess_id)
# Get all 555 neuron structure labels
unit_labels_full = data.units['ecephys_structure_acronym'].values  # length 555

units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- MAIN LOOP ---
for stim_label in stim_label_list:
    print(f"\n--- Processing stimulus: {stim_label} ---")
    ls = 60 if stim_label != 'flashes' else 3

    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            print(f" --> Missing file: {file}")
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units_idx = result.get('units', units_to_use_idx_[stim_label])
        trial_units = trial_units_idx

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError as e:
            print(f" --> Unit ID not found in global index: {e}")
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += ce[i, j]
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        print(f"!!! No valid trials for {stim_label}, skipping")
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials

    min_stability = 0.8
    mask = stability < min_stability
    avg_strength[mask] = 0
    stability[mask] = 0

    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]

    G = nx.DiGraph()
    for i in range(len(units)):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(len(units)):
        for j in range(len(units)):
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    print(f" --> Built graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

    pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
    fig, ax = plt.subplots(figsize=(12, 10))
    node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]

    weights = [np.abs(d['weight']) for _, _, d in G.edges(data=True)]
    stabilities = [d['stability'] for _, _, d in G.edges(data=True)]

        # NEW: Print max edge weight
    max_weight = max(weights) if weights else 0
    print(f" --> Max edge weight for {stim_label}: {max_weight:.4f}")
    min_alpha = 0.3
    max_width_scale = 1.0

    for (u, v, d) in G.edges(data=True):
        alpha_val = min_alpha + (1 - min_alpha) * d['stability']
        width_val = np.interp(np.abs(d['weight']), [min(weights), max(weights)], [0.1, max_width_scale])
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=width_val,
            alpha=alpha_val,
            edge_color='gray',aA
            arrows=True,
            ax=ax,
            connectionstyle='arc3,rad=0.2'
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    # Final plotting and legend setup
    subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
    legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
    ax.legend(handles=legend_patches, title='Brain Subregions', loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)

    fig.subplots_adjust(bottom=0.30, right=0.72)
    
    # Add edge thickness legend with mapped values
    arrow_legend_y = 0.80
    ax.annotate('Edge Thickness ~ |Weight|', xy=(1.01, arrow_legend_y), xycoords='axes fraction', fontsize=8)
    
    legend_weights = [min(weights), (min(weights) + max(weights)) / 2, max(weights)]
    legend_labels = [f"{w:.2f}" for w in legend_weights]
    legend_widths = [np.interp(w, [min(weights), max(weights)], [0.1, max_width_scale]) for w in legend_weights]
    
    for i, (w_val, w_lbl) in enumerate(zip(legend_widths, legend_labels)):
        y_pos = arrow_legend_y - (i + 1) * 0.04
        ax.annotate(
            '',
            xy=(1.07, y_pos), xytext=(1.03, y_pos),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', lw=w_val * 2, color='black')
        )
        ax.text(1.08, y_pos, w_lbl, transform=ax.transAxes, va='center', fontsize=8)
    
    fig.subplots_adjust(bottom=0.30, right=0.78)
    
    # Add edge transparency colorbar with numeric values
    cb_ax1 = fig.add_axes([0.79, 0.4, 0.01, 0.2])
    cb1 = plt.colorbar(plt.cm.ScalarMappable(cmap='Greys'), cax=cb_ax1)
    cb1.set_label('Stability (Edge Transparency)')
    cb1.set_ticks([min_alpha, 1.0])
    cb1.set_ticklabels([f"{min_alpha:.2f}", "1.00"])
    
    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials')
    ax.axis('off')
    
    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved graph: {basename}.pdf and .png")

In [ ]:
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.8  # NEW: Stability threshold used everywhere

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- FIRST PASS: Compute global weight min/max and store edge data ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])  # NEW: Take absolute value
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)
#     stim_edge_data[stim_label] = {
#     'avg_strength': avg_strength,
#     'stability': stability,
#     'n_trials': n_trials
# }

# --- SECOND PASS: Plot using global scale ---
max_width_scale = 1.0
min_alpha = 0.3

for stim_label, (avg_strength, stability, n_trials) in stim_edge_data.items():
    print(f"\n--- Processing stimulus: {stim_label} ---")

    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]

    G = nx.DiGraph()
    for i in range(len(units)):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(len(units)):
        for j in range(len(units)):
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    print(f" --> Built graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    weights = [np.abs(d['weight']) for _, _, d in G.edges(data=True)]
    max_w = max(weights) if weights else 0
    print(f" --> Max edge weight for {stim_label}: {max_w:.4f}")

    pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
    fig, ax = plt.subplots(figsize=(12, 10))
    node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]

    for (u, v, d) in G.edges(data=True):
        alpha_val = min_alpha + (1 - min_alpha) * d['stability']
        width_val = np.interp(np.abs(d['weight']), [global_min_weight, global_max_weight], [0.1, max_width_scale])
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=width_val,
            alpha=alpha_val,
            edge_color='gray',
            arrows=True,
            ax=ax,
            connectionstyle='arc3,rad=0.2'
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    # -- Legend: Brain Subregions (Moved Below) --
    subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
    legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
    ax.legend(
        handles=legend_patches,
        title='Brain Subregions',
        loc='lower center',
        bbox_to_anchor=(0.5, -0.32),
        ncol=4,
        fontsize=11,
        title_fontsize=12,
        frameon=False
    )

    # -- Edge Thickness Legend (Moved Below) --
    edge_legend_y_base = -0.42
    ax.annotate('Edge Thickness ~ |Weight|', xy=(0.5, edge_legend_y_base), xycoords='axes fraction', ha='center', fontsize=11)
    legend_weights = [global_min_weight, (global_min_weight + global_max_weight) / 2, global_max_weight]
    legend_labels = [f"{w:.2f}" for w in legend_weights]
    legend_widths = [np.interp(w, [global_min_weight, global_max_weight], [0.1, max_width_scale]) for w in legend_weights]

    for i, (w_val, w_lbl) in enumerate(zip(legend_widths, legend_labels)):
        x = 0.4 + i * 0.1
        y = edge_legend_y_base - 0.04
        ax.annotate(
            '', xy=(x + 0.02, y), xytext=(x, y),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', lw=w_val * 1.0, color='black')
        )
        ax.text(x + 0.03, y, w_lbl, transform=ax.transAxes, va='center', fontsize=10)

    # -- Stability Colorbar (Moved Below) --
    cb_ax1 = fig.add_axes([0.77, 0.05, 0.2, 0.02])  # Horizontal bar at the bottom
    cb1 = plt.colorbar(plt.cm.ScalarMappable(cmap='Greys'), cax=cb_ax1, orientation='horizontal')
    cb1.set_label('Stability (Edge Transparency)', fontsize=11)
    cb1.ax.tick_params(labelsize=10)
    cb1.set_ticks([min_alpha, 1.0])
    cb1.set_ticklabels([f"{min_alpha:.2f}", "1.00"])

    # -- Adjust layout to fit all bottom elements --
    fig.subplots_adjust(bottom=0.45, right=0.75)

    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials', fontsize=13)
    ax.axis('off')

    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved graph: {basename}.pdf and .png")



In [ ]:
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.8  # NEW: Stability threshold used everywhere

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- FIRST PASS: Compute global weight min/max and store edge data ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])  # NEW: Take absolute value
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)
#     stim_edge_data[stim_label] = {
#     'avg_strength': avg_strength,
#     'stability': stability,
#     'n_trials': n_trials
# }

# --- SECOND PASS: Plot using global scale ---
max_width_scale = 1.0
min_alpha = 0.3

for stim_label, (avg_strength, stability, n_trials) in stim_edge_data.items():
    print(f"\n--- Processing stimulus: {stim_label} ---")

    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]

    G = nx.DiGraph()
    for i in range(len(units)):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(len(units)):
        for j in range(len(units)):
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    print(f" --> Built graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    weights = [np.abs(d['weight']) for _, _, d in G.edges(data=True)]
    max_w = max(weights) if weights else 0
    print(f" --> Max edge weight for {stim_label}: {max_w:.4f}")

    pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
    fig, ax = plt.subplots(figsize=(12, 10))
    node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]

    for (u, v, d) in G.edges(data=True):
        alpha_val = min_alpha + (1 - min_alpha) * d['stability']
        width_val = 2*np.interp(np.abs(d['weight']), [global_min_weight, global_max_weight], [0.1, max_width_scale])
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=width_val,
            alpha=alpha_val,
            edge_color='gray',
            arrows=True,
            ax=ax,
            connectionstyle='arc3,rad=0.2'
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
    legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
    ax.legend(handles=legend_patches, title='Brain Subregions', loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)

    fig.subplots_adjust(bottom=0.30, right=0.72)

    # Edge thickness legend (now consistent globally)
    arrow_legend_y = 0.80
    ax.annotate('Edge Thickness ~ |Weight|', xy=(1.01, arrow_legend_y), xycoords='axes fraction', fontsize=8)
    legend_weights = [global_min_weight, (global_min_weight + global_max_weight) / 2, global_max_weight]
    legend_labels = [f"{w:.2f}" for w in legend_weights]
    legend_widths = [np.interp(w, [global_min_weight, global_max_weight], [0.1, max_width_scale]) for w in legend_weights]

    for i, (w_val, w_lbl) in enumerate(zip(legend_widths, legend_labels)):
        y_pos = arrow_legend_y - (i + 1) * 0.04
        ax.annotate(
            '',
            xy=(1.07, y_pos), xytext=(1.03, y_pos),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', lw=w_val * 2.0, color='black')
        )
        ax.text(1.08, y_pos, w_lbl, transform=ax.transAxes, va='center', fontsize=8)


    fig.subplots_adjust(bottom=0.30, right=0.78)

    # Colorbar for edge transparency
    cb_ax1 = fig.add_axes([0.79, 0.4, 0.01, 0.2])
    cb1 = plt.colorbar(plt.cm.ScalarMappable(cmap='Greys'), cax=cb_ax1)
    cb1.set_label('Stability (Edge Transparency)')
    cb1.set_ticks([0.0, 0.5, 0.8, 1.0])
    cb1.set_ticklabels(["0.0", "0.5", "0.8", "1.0"])

    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials')
    ax.axis('off')

    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved graph: {basename}.pdf and .png")


In [ ]:
# --- BAR & BOX PLOTS FOR MULTIPLE STABILITY THRESHOLDS ---
cutoff_fraction = 0.10
stability_thresholds = [0.8]#[0.5, 0.8]

region_labels = unit_labels_permuted
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(region_labels):
    region_neuron_indices[label].append(idx)

# Filter small regions
filtered_region_labels = [r for r in region_neuron_indices if len(region_neuron_indices[r]) > 5]

for stab_thresh in stability_thresholds:

    # --- Bar Plot ---
    bar_data = []
    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][0]
        stability = stim_edge_data[stim_label][1]
        for i in range(len(units)):
            for j in range(len(units)):
                source = region_labels[i]
                target = region_labels[j]
                if source not in filtered_region_labels or target not in filtered_region_labels:
                    continue
                w = avg_strength[i, j]
                s = stability[i, j]
                if w == 0 or s < stab_thresh:
                    continue
                bar_data.append({
                    'Stimulus': stim_label,
                    'Source': source,
                    'Target': target,
                    'Connection': f"{source} → {target}",
                    'Weight': w,
                    'Stability': s
                })

    bar_df = pd.DataFrame(bar_data)
    if not bar_df.empty:
        grouped = bar_df.groupby(['Stimulus', 'Source', 'Connection']).agg(
            Count=('Connection', 'size'),
            Stability_Mean=('Stability', 'mean')
        ).reset_index()

        # Normalize to percentage
        source_counts = {src: len(region_neuron_indices[src]) for src in filtered_region_labels}
        grouped['Percentage'] = grouped.apply(lambda row: 100 * row['Count'] / (source_counts.get(row['Source'], 1)**2), axis=1)

        plt.figure(figsize=(14, 6))
        sns.barplot(data=grouped, x='Connection', y='Percentage', hue='Stimulus', ci=None)
        plt.xticks(rotation=90)
        plt.title(f"% Stable Connections ≥ {stab_thresh}")
        plt.ylabel('% of Possible Neuron Pairs')
        plt.tight_layout()
        plt.savefig(f"{plot_save_directory}/barplot_percent_stab{int(stab_thresh*100)}.png", dpi=300)
        plt.show()
        plt.close()

    # --- Box Plot ---
    region_pair_counts = defaultdict(lambda: defaultdict(int))
    region_pair_weights = defaultdict(lambda: defaultdict(list))

    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][0]
        stability = stim_edge_data[stim_label][1]
        for i in range(len(units)):
            for j in range(len(units)):
                source = region_labels[i]
                target = region_labels[j]
                if source not in filtered_region_labels or target not in filtered_region_labels:
                    continue
                w = avg_strength[i, j]
                s = stability[i, j]
                if w == 0 or s < stab_thresh:
                    continue
                region_pair_counts[stim_label][(source, target)] += 1
                region_pair_weights[stim_label][(source, target)].append(w)

    eligible_pairs = set()
    for stim_label in stim_label_list:
        for (source, target), count in region_pair_counts[stim_label].items():
            possible = len(region_neuron_indices[source]) * len(region_neuron_indices[target])
            if possible == 0:
                continue
            if count / possible >= cutoff_fraction:
                eligible_pairs.add((source, target))

    box_data = []
    for stim_label in stim_label_list:
        for (source, target) in eligible_pairs:
            for w in region_pair_weights[stim_label].get((source, target), []):
                box_data.append({
                    'Stimulus': stim_label,
                    'Region Pair': f"{source} → {target}",
                    'Edge Weight': w
                })

    box_df = pd.DataFrame(box_data)
    if not box_df.empty:
        plt.figure(figsize=(14, 6))
        sns.boxplot(data=box_df, x='Region Pair', y='Edge Weight', hue='Stimulus')
        plt.xticks(rotation=90)
        plt.title(f"Edge Weights (Dense Region Pairs, Stability ≥ {stab_thresh})")
        plt.tight_layout()
        plt.savefig(f"{plot_save_directory}/boxplot_edge_weights_stab{int(stab_thresh*100)}.png", dpi=300)
        plt.show()
        plt.close()

In [ ]:
# --- BAR, BOX & GRAPH PLOTS FOR MULTIPLE STABILITY THRESHOLDS ---
from collections import defaultdict

cutoff_fraction = 0.10
stability_thresholds = [0.8]#[0.5, 0.8]

region_labels = unit_labels_permuted
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(region_labels):
    region_neuron_indices[label].append(idx)

# Filter small regions
filtered_region_labels = [r for r in region_neuron_indices if len(region_neuron_indices[r]) > 5]

for stab_thresh in stability_thresholds:

    # # --- Graph Plot ---
    # for stim_label in stim_label_list:
    #     avg_strength = stim_edge_data[stim_label][0]
    #     stability = stim_edge_data[stim_label][1]

    #     G = nx.DiGraph()
    #     for i in range(len(units)):
    #         label = unit_labels_permuted[i]
    #         G.add_node(i, area=label)

    #     for i in range(len(units)):
    #         for j in range(len(units)):
    #             source = region_labels[i]
    #             target = region_labels[j]
    #             if source not in filtered_region_labels or target not in filtered_region_labels:
    #                 continue
    #             w = avg_strength[i, j]
    #             s = stability[i, j]
    #             if w != 0 and s >= stab_thresh:
    #                 G.add_edge(i, j, weight=w, stability=s)

    #     pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
    #     fig, ax = plt.subplots(figsize=(12, 10))
    #     node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]
    #     weights = [np.abs(d['weight']) for _, _, d in G.edges(data=True)]
    #     stabilities = [d['stability'] for _, _, d in G.edges(data=True)]
    #     weights_min, weights_max = global_min_weight, global_max_weight

    #     min_alpha = 0.1
    #     global_min_stability = min(stabilities) if stabilities else 0.0
    #     global_max_stability = max(stabilities) if stabilities else 1.0
    #     max_width_scale = 1.0

    #     for (u, v, d) in G.edges(data=True):
    #         alpha_val = np.interp(d['stability'], [global_min_stability, global_max_stability], [min_alpha, 1.0])
    #         width_val = 2*np.interp(np.abs(d['weight']), [weights_min, weights_max], [0.1, max_width_scale])
    #         nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=width_val, alpha=alpha_val,
    #                                edge_color='gray', arrows=True, ax=ax, connectionstyle='arc3,rad=0.2')

    #     nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    #     nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)
    #     subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
    #     legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
    #     ax.legend(handles=legend_patches, title='Brain Subregions', loc='upper center', bbox_to_anchor=(0.5, -0.12),
    #               ncol=4, fontsize=8, frameon=False)
    #     fig.subplots_adjust(bottom=0.30, right=0.72)
        
    #     # Add edge thickness legend
    #     arrow_legend_y = 0.80
    #     ax.annotate('Edge Thickness ~ |Weight|', xy=(1.01, arrow_legend_y), xycoords='axes fraction', fontsize=8)
    #     legend_weights = [weights_min, (weights_min + weights_max) / 2, weights_max]
    #     legend_labels = [f"{w:.2f}" for w in legend_weights]
    #     legend_widths = [np.interp(w, [weights_min, weights_max], [0.1, 1.0]) for w in legend_weights]
    #     for i, (w_val, w_lbl) in enumerate(zip(legend_widths, legend_labels)):
    #         y_pos = arrow_legend_y - (i + 1) * 0.04
    #         ax.annotate('', xy=(1.07, y_pos), xytext=(1.03, y_pos),
    #                     xycoords='axes fraction', textcoords='axes fraction',
    #                     arrowprops=dict(arrowstyle='->', lw=w_val, color='black'))
    #         ax.text(1.08, y_pos, w_lbl, transform=ax.transAxes, va='center', fontsize=8)
    #     # Add stability colorbar
    #     cb_ax1 = fig.add_axes([0.79, 0.4, 0.01, 0.2])
    #     sm = plt.cm.ScalarMappable(cmap='Greys')
    #     sm.set_clim(min_alpha, 1.0)
    #     cb1 = plt.colorbar(sm, cax=cb_ax1)
    #     cb1.set_label('Stability (Edge Transparency)')
    #     cb_ax1.text(0.5, -0.05, 'Less Stable', rotation=0, va='top', ha='center', transform=cb_ax1.transAxes, fontsize=7)
    #     cb_ax1.text(0.5, 1.05, 'More Stable', rotation=0, va='bottom', ha='center', transform=cb_ax1.transAxes, fontsize=7)
    #     cb1.set_ticks([min_alpha, 1.0])
    #     cb1.set_ticklabels([f"{min_alpha:.2f}", "1.00"])
        

    #     ax.set_title(f"{stim_label} – Graph (Stability ≥ {stab_thresh})")
    #     ax.axis('off')
    #     fig.savefig(f"{plot_save_directory}/graph_{stim_label}_stab{int(stab_thresh * 100)}.png", dpi=300, bbox_inches='tight')
    #     plt.show()
    #     plt.close()

    # --- Bar Plot ---
    bar_data = []
    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][0]
        stability = stim_edge_data[stim_label][1]
        for i in range(len(units)):
            for j in range(len(units)):
                source = region_labels[i]
                target = region_labels[j]
                if source not in filtered_region_labels or target not in filtered_region_labels:
                    continue
                w = avg_strength[i, j]
                s = stability[i, j]
                if w == 0 or s < stab_thresh:
                    continue
                bar_data.append({
                    'Stimulus': stim_label,
                    'Source': source,
                    'Target': target,
                    'Connection': f"{source} → {target}",
                    'Weight': w,
                    'Stability': s
                })

    bar_df = pd.DataFrame(bar_data)
    if not bar_df.empty:
        grouped = bar_df.groupby(['Stimulus', 'Source', 'Connection']).agg(
            Count=('Connection', 'size'),
            Stability_Mean=('Stability', 'mean')
        ).reset_index()

        # Normalize to percentage
        source_counts = {src: len(region_neuron_indices[src]) for src in filtered_region_labels}
        grouped['Percentage'] = grouped.apply(lambda row: 100 * row['Count'] / (source_counts.get(row['Source'], 1)**2), axis=1)

        plt.figure(figsize=(14, 6))
        sns.barplot(data=grouped, x='Connection', y='Percentage', hue='Stimulus', ci=None)
        plt.xticks(rotation=90)
        plt.title(f"% Stable Connections ≥ {stab_thresh}")
        plt.ylabel('% of Possible Neuron Pairs')
        plt.tight_layout()
        plt.savefig(f"{plot_save_directory}/barplot_percent_stab{int(stab_thresh*100)}.png", dpi=300)
        plt.show()
        plt.close()

    # # --- Box Plot ---
    # region_pair_counts = defaultdict(lambda: defaultdict(int))
    # region_pair_weights = defaultdict(lambda: defaultdict(list))

    # for stim_label in stim_label_list:
    #     avg_strength = stim_edge_data[stim_label][0]
    #     stability = stim_edge_data[stim_label][1]
    #     for i in range(len(units)):
    #         for j in range(len(units)):
    #             source = region_labels[i]
    #             target = region_labels[j]
    #             if source not in filtered_region_labels or target not in filtered_region_labels:
    #                 continue
    #             w = avg_strength[i, j]
    #             s = stability[i, j]
    #             if w == 0 or s < stab_thresh:
    #                 continue
    #             region_pair_counts[stim_label][(source, target)] += 1
    #             region_pair_weights[stim_label][(source, target)].append(w)

    # eligible_pairs = set()
    # for stim_label in stim_label_list:
    #     for (source, target), count in region_pair_counts[stim_label].items():
    #         possible = len(region_neuron_indices[source]) * len(region_neuron_indices[target])
    #         if possible == 0:
    #             continue
    #         if count / possible >= cutoff_fraction:
    #             eligible_pairs.add((source, target))

    # box_data = []
    # for stim_label in stim_label_list:
    #     for (source, target) in eligible_pairs:
    #         for w in region_pair_weights[stim_label].get((source, target), []):
    #             box_data.append({
    #                 'Stimulus': stim_label,
    #                 'Region Pair': f"{source} → {target}",
    #                 'Edge Weight': w
    #             })

    # box_df = pd.DataFrame(box_data)
    # if not box_df.empty:
    #     plt.figure(figsize=(14, 6))
    #     sns.boxplot(data=box_df, x='Region Pair', y='Edge Weight', hue='Stimulus')
    #     plt.xticks(rotation=90)
    #     plt.title(f"Edge Weights (Dense Region Pairs, Stability ≥ {stab_thresh})")
    #     plt.tight_layout()
    #     plt.savefig(f"{plot_save_directory}/boxplot_edge_weights_stab{int(stab_thresh*100)}.png", dpi=300)
    #     plt.show()
    #     plt.close()


In [ ]:
##Differentiated by source regions
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.8  # NEW: Stability threshold used everywhere

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- FIRST PASS: Compute global weight min/max and store edge data ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])  # NEW: Take absolute value
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)
    
##differentiated by source regions
# --- BAR & BOX PLOTS FOR MULTIPLE STABILITY THRESHOLDS ---
cutoff_fraction = 0.10
stability_thresholds = [0.8]#[0.5, 0.8]

region_labels = unit_labels_permuted
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(region_labels):
    region_neuron_indices[label].append(idx)

# Filter small regions
filtered_region_labels = [r for r in region_neuron_indices if len(region_neuron_indices[r]) > 5]

# Broad region mapping
broad_region_map = {
    'VISp': 'Visual cortex', 'VISl': 'Visual cortex', 'VISrl': 'Visual cortex',
    'VISal': 'Visual cortex', 'VISpm': 'Visual cortex', 'VISam': 'Visual cortex',
    'CA1': 'Hippocampus', 'CA2': 'Hippocampus', 'CA3': 'Hippocampus',
    'DG': 'Hippocampus', 'SUB': 'Hippocampus',
    'POL': 'Other', 'LGv': 'Thalamus', 'LP': 'Thalamus'
}

# Get list of present broad regions based on filtered fine regions
broad_regions_present = sorted(set(broad_region_map.get(r, 'Other') for r in filtered_region_labels))

for stab_thresh in stability_thresholds:

    # --- Bar Plot ---
    bar_data = []
    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][0]
        stability = stim_edge_data[stim_label][1]
        for i in range(len(units)):
            for j in range(len(units)):
                source = region_labels[i]
                target = region_labels[j]
                if source not in filtered_region_labels or target not in filtered_region_labels:
                    continue
                w = avg_strength[i, j]
                s = stability[i, j]
                if w == 0 or s < stab_thresh:
                    continue
                bar_data.append({
                    'Stimulus': stim_label,
                    'Source': source,
                    'Target': target,
                    'Connection': f"{source} → {target}",
                    'Weight': w,
                    'Stability': s
                })

    bar_df = pd.DataFrame(bar_data)

    if not bar_df.empty:
        grouped = bar_df.groupby(['Stimulus', 'Source', 'Connection']).agg(
            Count=('Connection', 'size'),
            Stability_Mean=('Stability', 'mean')
        ).reset_index()

        # Normalize to percentage
        source_counts = {src: len(region_neuron_indices[src]) for src in filtered_region_labels}
        grouped['Percentage'] = grouped.apply(
            lambda row: 100 * row['Count'] / (source_counts.get(row['Source'], 1)**2),
            axis=1
        )
        max_y = grouped['Percentage'].max()
        from matplotlib.lines import Line2D
        import numpy as np
        import matplotlib.pyplot as plt
        
        # Correct stimulus-to-color mapping
        stim_color_map = {
            'flashes': '#1f77b4',           # 🔵
            'natural_scenes': '#ff7f0e',    # 🟧
            'static_gratings': '#2ca02c',   # 🟢
            'gabors': '#d62728'             # 🔴
        }
        ordered_stimuli = ['flashes', 'natural_scenes', 'static_gratings', 'gabors']
        
        bar_width = 0.2
        group_spacing = 1.0
        
        for broad_source in broad_regions_present:
            # Total neurons in all regions of this broad group
            included_sources_all = [r for r in region_neuron_indices if broad_region_map.get(r, 'Other') == broad_source]
            total_neurons = sum(len(region_neuron_indices[r]) for r in included_sources_all)
        
            # Use only filtered (large) regions for plotting
            included_sources_filtered = [r for r in filtered_region_labels if broad_region_map.get(r, 'Other') == broad_source]
            sub_group = grouped[grouped['Source'].isin(included_sources_filtered)]
            if sub_group.empty:
                continue
        
            sub_group = sub_group[sub_group['Percentage'] > 0]
            connections = sorted(sub_group['Connection'].unique())
        
            x_positions = []
            heights = []
            yerrs = []
            colors = []
            labels = []
        
            for group_index, conn in enumerate(connections):
                conn_data = sub_group[sub_group['Connection'] == conn]
                stim_bars = []
                for stim in ordered_stimuli:
                    row = conn_data[conn_data['Stimulus'] == stim]
                    if not row.empty:
                        pct = row['Percentage'].values[0]
                        count = row['Count'].values[0]
                        n_trials = stim_edge_data[stim][2]
        
                        if n_trials > 0:
                            p = count / n_trials
                            se_pct = np.sqrt(p * (1 - p) / n_trials) * 100
                            if not np.isfinite(se_pct):
                                se_pct = 0.0
                        else:
                            se_pct = 0.0
        
                        stim_bars.append((stim, pct, se_pct))
        
                n_bars = len(stim_bars)
                group_start = group_index * group_spacing
        
                for i, (stim, pct, se) in enumerate(stim_bars):
                    xpos = group_start + (i - n_bars / 2) * bar_width + bar_width / 2
                    x_positions.append(xpos)
                    heights.append(pct)
                    yerrs.append(se)
                    colors.append(stim_color_map[stim])
                    labels.append(conn)
        
            # Deduplicate x-axis labels
            final_labels = []
            last_conn = ""
            for label in labels:
                if label == last_conn:
                    final_labels.append("")
                else:
                    final_labels.append(label)
                    last_conn = label
        
            # Plot
            plt.figure(figsize=(max(10, len(x_positions) * 0.3), 6))
            plt.bar(x_positions, heights, capsize=5,
                    color=colors, width=bar_width)
        
            plt.xticks(x_positions, final_labels, rotation=90)
            plt.ylabel('% of Possible Neuron Pairs')
        
            top = max([h + e for h, e in zip(heights, yerrs) if np.isfinite(h) and np.isfinite(e)])
            plt.ylim(0, top * 1.15)
        
            plt.title(f"{broad_source} (n={total_neurons} neurons): % Stable Connections ≥ {stab_thresh}")
            plt.tight_layout()
        
            # Legend inside top-right
            legend_elements = [
                Line2D([0], [0], color=stim_color_map[stim], lw=6, label=stim)
                for stim in ordered_stimuli
            ]
            plt.legend(
                handles=legend_elements,
                title="Stimulus",
                loc='upper right',
                frameon=True,
                fontsize='medium',
                title_fontsize='medium',
                borderpad=0.6,
                labelspacing=0.4
            )
        
            outname = f"{plot_save_directory}/barplot_percent_{broad_source.replace(' ', '_')}_grouped_clean_stab{int(stab_thresh*100)}_err.png"
            plt.savefig(outname, dpi=300)
            plt.close()



# Get present broad source regions
broad_regions_present = sorted(set(broad_region_map.get(r, 'Other') for r in filtered_region_labels))

In [ ]:
##Differentiated by source regions V2 - error bars
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.8  # NEW: Stability threshold used everywhere

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- FIRST PASS: Compute global weight min/max and store edge data ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])  # NEW: Take absolute value
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)
    
##differentiated by source regions
# --- BAR & BOX PLOTS FOR MULTIPLE STABILITY THRESHOLDS ---
cutoff_fraction = 0.10
stability_thresholds = [0.8]#[0.5, 0.8]

region_labels = unit_labels_permuted
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(region_labels):
    region_neuron_indices[label].append(idx)

# Filter small regions
filtered_region_labels = [r for r in region_neuron_indices if len(region_neuron_indices[r]) > 5]

# Broad region mapping
broad_region_map = {
    'VISp': 'Visual cortex', 'VISl': 'Visual cortex', 'VISrl': 'Visual cortex',
    'VISal': 'Visual cortex', 'VISpm': 'Visual cortex', 'VISam': 'Visual cortex',
    'CA1': 'Hippocampus', 'CA2': 'Hippocampus', 'CA3': 'Hippocampus',
    'DG': 'Hippocampus', 'SUB': 'Hippocampus',
    'POL': 'Other', 'LGv': 'Thalamus', 'LP': 'Thalamus'
}

# Get list of present broad regions based on filtered fine regions
broad_regions_present = sorted(set(broad_region_map.get(r, 'Other') for r in filtered_region_labels))

###trial stuff
# === Trial-wise extraction ===
trialwise = []
for stim in stim_label_list:
    ls = 60 if stim != 'flashes' else 3
    if stim not in units_to_use_idx_:
        continue
    idx_subset = units_to_use_idx_[stim]
    for idx in range(ls):
        fname = f"{save_directory}/ID{sess_id}_{stim}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(fname): continue
        with open(fname, 'rb') as f:
            ce = pkl.load(f)['ce']
        ce = ce - np.diag(np.diag(ce))
        if ce.shape[0] != len(idx_subset):
            continue
        trial_labels = [unit_labels_full[i] for i in idx_subset]
        for i, src in enumerate(trial_labels):
            for j, tgt in enumerate(trial_labels):
                if src == tgt: continue
                if src not in filtered_region_labels or tgt not in filtered_region_labels: continue
                conn = f"{src} → {tgt}"
                broad_src = broad_region_map.get(src, 'Other')
                stable = 1 if ce[i, j] != 0 else 0
                trialwise.append({
                    'Stimulus': stim,
                    'Trial': idx,
                    'BroadSource': broad_src,
                    'Connection': conn,
                    'Source': src,
                    'Target': tgt,
                    'Stable': stable
                })

trial_df = pd.DataFrame(trialwise)
grouped_trials = (
    trial_df
    .groupby(['Stimulus', 'Trial', 'BroadSource', 'Connection', 'Source'])
    .agg(StableCount=('Stable', 'sum'), Total=('Stable', 'count'))
    .reset_index()
)
grouped_trials['Percentage'] = 100 * grouped_trials['StableCount'] / grouped_trials['Total']

# === Compute mean, SEM, and filter based on threshold ===
summary_df = (
    grouped_trials
    .groupby(['Stimulus', 'Connection', 'BroadSource', 'Source'])
    .agg(
        MeanPercentage=('Percentage', 'mean'),
        SEM=('Percentage', lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
        Count=('StableCount', 'count'),
        AboveThresh=('Percentage', lambda x: (x >= stab_thresh * 100).sum())
    )
    .reset_index()
)

# === Final Filter: include only connections where ≥stab_thresh of trials were stable ===
# Figure out the correct total-trials-per-stimulus here; e.g. 60 or 3
n_trials = {'natural_scenes':60, 'static_gratings':60, 'gabors':60, 'flashes':3}
summary_df = summary_df[
    summary_df.apply(lambda row: row['AboveThresh'] >= stab_thresh * n_trials[row['Stimulus']], axis=1)
]

# === Plotting ===
bar_width = 0.2
group_spacing = 1.0
os.makedirs(plot_save_directory, exist_ok=True)

for broad_source in sorted(summary_df['BroadSource'].unique()):
    print(f"Generating plot for {broad_source}…")
    import os
    print("Current working directory:", os.getcwd())
    sub = summary_df[summary_df['BroadSource'] == broad_source]
    if sub.empty: continue
    total_neurons = sum(len(region_neuron_indices[r]) for r in region_neuron_indices if broad_region_map.get(r) == broad_source)

    connections = sorted(sub['Connection'].unique())
    x_pos, heights, yerrs, colors, labels = [], [], [], [], []
    for i_conn, conn in enumerate(connections):
        conn_df = sub[sub['Connection'] == conn]
        group_start = i_conn * group_spacing
        stim_bars = []
        for stim in ordered_stimuli:
            row = conn_df[conn_df['Stimulus'] == stim]
            if not row.empty:
                h = row['MeanPercentage'].values[0]
                e = row['SEM'].values[0]
                stim_bars.append((stim, h, e))
        n_bars = len(stim_bars)
        for i, (stim, h, e) in enumerate(stim_bars):
            xpos = group_start + (i - n_bars / 2) * bar_width + bar_width / 2
            x_pos.append(xpos)
            heights.append(h)
            yerrs.append(e)
            colors.append(stim_color_map[stim])
            labels.append(conn)

    # Clean X labels
    clean_labels = []
    last = ""
    for lbl in labels:
        clean_labels.append(lbl if lbl != last else "")
        last = lbl

    # plt.figure(figsize=(max(10, len(x_pos)*0.3), 6))
    # plt.bar(x_pos, heights, yerr=yerrs, color=colors, width=bar_width, capsize=4)
    # plt.xticks(x_pos, clean_labels, rotation=90)
    # plt.ylabel('% of Possible Neuron Pairs')
    # plt.title(f"{broad_source} (n={total_neurons} neurons): Connections Stable in ≥ {int(stab_thresh*100)}% Trials")
    # plt.tight_layout()
    # legend_handles = [Line2D([0], [0], color=stim_color_map[s], lw=6, label=s) for s in ordered_stimuli]
    # plt.legend(handles=legend_handles, title='Stimulus', loc='upper right')
    # fname = f"{plot_save_directory}/barplot_percent_{broad_source.replace(' ', '_')}_grouped_clean_stab{int(stab_thresh*100)}_sem_thresh_applied.png"
    # plt.savefig(fname, dpi=300)
    # plt.close()


# from matplotlib.lines import Line2D

# stab_thresh = 0.8

# # Constants
# stim_color_map = {
#     'flashes': '#1f77b4',
#     'natural_scenes': '#ff7f0e',
#     'static_gratings': '#2ca02c',
#     'gabors': '#d62728'
# }
# ordered_stimuli = ['flashes', 'natural_scenes', 'static_gratings', 'gabors']
# bar_width = 0.2
# group_spacing = 1.0

# for broad_source in broad_regions_present:
#     # Total neuron count in this broad region
#     included_sources_all = [r for r in region_neuron_indices if broad_region_map.get(r, 'Other') == broad_source]
#     total_neurons = sum(len(region_neuron_indices[r]) for r in included_sources_all)

#     # Filter only relevant rows from new summary_df
#     sub_group = summary_df[summary_df['BroadSource'] == broad_source]
#     if sub_group.empty:
#         continue

#     connections = sorted(sub_group['Connection'].unique())

#     x_positions = []
#     heights = []
#     yerrs = []
#     colors = []
#     labels = []

#     for group_index, conn in enumerate(connections):
#         conn_data = sub_group[sub_group['Connection'] == conn]
#         stim_bars = []
#         for stim in ordered_stimuli:
#             row = conn_data[conn_data['Stimulus'] == stim]
#             if not row.empty:
#                 pct = row['MeanPercentage'].values[0]
#                 se = row['SEM'].values[0]
#                 stim_bars.append((stim, pct, se))

#         n_bars = len(stim_bars)
#         group_start = group_index * group_spacing

#         for i, (stim, pct, se) in enumerate(stim_bars):
#             xpos = group_start + (i - n_bars / 2) * bar_width + bar_width / 2
#             x_positions.append(xpos)
#             heights.append(pct)
#             yerrs.append(se)
#             colors.append(stim_color_map[stim])
#             labels.append(conn)

#     # Clean labels
#     final_labels = []
#     last_conn = ""
#     for label in labels:
#         if label == last_conn:
#             final_labels.append("")
#         else:
#             final_labels.append(label)
#             last_conn = label

#     # Plotting
#     plt.figure(figsize=(max(10, len(x_positions) * 0.3), 6))
#     plt.bar(x_positions, heights, yerr=yerrs, capsize=5,
#             color=colors, width=bar_width)

#     plt.xticks(x_positions, final_labels, rotation=90)
#     plt.ylabel('% of Possible Neuron Pairs')
#     top = max([h + e for h, e in zip(heights, yerrs) if np.isfinite(h) and np.isfinite(e)])
#     plt.ylim(0, top * 1.15)
#     plt.title(f"{broad_source} (n={total_neurons} neurons): Connections Stable in ≥ {int(stab_thresh*100)}% Trials")
#     plt.tight_layout()

#     # Legend inside top-right
#     legend_elements = [
#         Line2D([0], [0], color=stim_color_map[stim], lw=6, label=stim)
#         for stim in ordered_stimuli
#     ]
#     plt.legend(
#         handles=legend_elements,
#         title="Stimulus",
#         loc='upper right',
#         frameon=True,
#         fontsize='medium',
#         title_fontsize='medium',
#         borderpad=0.6,
#         labelspacing=0.4
#     )

#     # Save
#     outname = f"{plot_save_directory}/barplot_percent_{broad_source.replace(' ', '_')}_grouped_clean_stab{int(stab_thresh*100)}_sem.png"
#     plt.savefig(outname, dpi=300)
#     plt.close()

In [ ]:
from collections import defaultdict
import os
import pickle as pkl
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Precompute region‐size normalization
source_counts = {
    src: len(region_neuron_indices[src])
    for src in filtered_region_labels
}

ordered_stimuli = ['flashes', 'natural_scenes', 'static_gratings', 'gabors']
stim_color_map = {
    'flashes':        '#1f77b4',
    'natural_scenes': '#ff7f0e',
    'static_gratings':'#2ca02c',
    'gabors':         '#d62728'
}

# 1) Gather per‐replicate percentages
rep_perc = defaultdict(list)
for stim in stim_label_list:
    ls = 3 if stim=='flashes' else 60
    unit_to_full = {u:i for i,u in enumerate(units)}
    for idx in range(ls):
        path = (
            f"{save_directory}/"
            f"ID{sess_id}_{stim}_bin{bin_size}"
            f"_alpha{alpha}_idx{idx}_niter{niter}.p"
        )
        if not os.path.exists(path):
            continue
        with open(path,'rb') as f:
            result = pkl.load(f)
        ce = result['ce']
        np.fill_diagonal(ce, 0)
        trial_units = result.get('units', units_to_use_idx_[stim])
        try:
            _ = [unit_to_full[u] for u in trial_units]
        except KeyError:
            continue

        trial_labels = [unit_labels_full[u] for u in trial_units]
        for source in filtered_region_labels:
            for target in filtered_region_labels:
                conn = f"{source} → {target}"
                src_pos = [i for i,lab in enumerate(trial_labels) if lab==source]
                tgt_pos = [j for j,lab in enumerate(trial_labels) if lab==target]
                if not src_pos or not tgt_pos:
                    continue
                cnt = sum(np.count_nonzero(ce[i, tgt_pos]) for i in src_pos)
                pct = cnt / (source_counts[source]**2) * 100
                rep_perc[(stim, conn)].append(pct)

# 2) Compute mean & SEM for stable connections
stats = {}
for (stim, conn), arr in rep_perc.items():
    arr = np.array(arr)
    presence = (arr>0).sum() / len(arr)
    if presence < min_stability:
        continue
    stats[(stim, conn)] = {
        'mean': arr.mean(),
        'sem':  arr.std(ddof=1) / np.sqrt(len(arr))
    }

# 3) Determine a common y‐axis upper limit (rounded to next 10)
all_vals = [v['mean'] + v['sem'] for v in stats.values()]
y_max = math.ceil(max(all_vals) / 10) * 10

# --- Plot parameters ---
bar_px        = 30      # pixel width per bar & per gap
dpi           = plt.rcParams['figure.dpi']
fig_height_in = 8       # fixed height (inches)
bar_w         = 1       # one slot per bar (data units)
group_gap     = 1       # one slot between groups
margin_in     = 1.0     # left+right margin (inches)

for broad in sorted({broad_region_map[r] for r in filtered_region_labels}):
    # build & sort connections for this region
    raw_conns = {
        conn
        for (stim, conn) in stats
        if broad_region_map[conn.split(' → ')[0]] == broad
    }
    conns = sorted(raw_conns, key=lambda c: tuple(c.split(' → ')))
    if not conns:
        continue

    # count bars per group
    bars_per_group = [
        sum((stim, conn) in stats for stim in ordered_stimuli)
        for conn in conns
    ]

    # total slots = bars + gaps
    n_bars_total = sum(bars_per_group)
    n_gap_slots  = len(conns) - 1
    total_slots  = n_bars_total + n_gap_slots

    # compute figure width so each slot = bar_px pixels
    inch_per_slot = bar_px / dpi
    fig_width_in  = total_slots * inch_per_slot + margin_in

    # start figure with fixed height
    plt.figure(figsize=(fig_width_in, fig_height_in), dpi=dpi)

    # cumulative group starts
    group_starts = []
    cursor = 0.0
    for n in bars_per_group:
        group_starts.append(cursor)
        cursor += n * bar_w + group_gap

    # collect bar positions & values
    xs, ys, es, cs = [], [], [], []
    for i, conn in enumerate(conns):
        start = group_starts[i]
        present = [s for s in ordered_stimuli if (s, conn) in stats]
        for j, stim in enumerate(present):
            m = stats[(stim, conn)]['mean']
            s = stats[(stim, conn)]['sem']
            xs.append(start + j * bar_w)
            ys.append(m)
            es.append(s)
            cs.append(stim_color_map[stim])

    # plot
    plt.bar(xs, ys, width=bar_w, yerr=es, capsize=4, color=cs)

    # common y‐scale
    plt.ylim(0, y_max)

    # xticks at group centers
    tick_pos = [
        start + (n * bar_w) / 2
        for start, n in zip(group_starts, bars_per_group)
    ]
    plt.xticks(tick_pos, conns, rotation=90)
    plt.tick_params(axis='x', which='both', length=0)

    # labels & title
    n_neurons = sum(
        len(region_neuron_indices[r])
        for r in filtered_region_labels
        if broad_region_map[r] == broad
    )
    plt.ylabel('% of Possible Neuron Pairs')
    plt.title(
        f"{broad} (n={n_neurons} neurons): "
        f"% Stable Connections ≥ {int(min_stability*100)}%"
    )

    # legend
    handles = [
        Line2D([0], [0], color=stim_color_map[s], lw=6, label=s)
        for s in ordered_stimuli
    ]
    loc    = 'upper right' if broad=='Hippocampus' else 'upper left'
    anchor = (0.98, 0.95) if broad=='Hippocampus' else (0.02, 0.95)
    plt.legend(handles=handles, title='Stimulus',
               loc=loc, bbox_to_anchor=anchor,
               frameon=True, framealpha=0.9)

    # save
    plt.tight_layout()
    out = os.path.join(
        plot_save_directory,
        f"bar_{broad.replace(' ', '_')}_sem.png"
    )
    plt.savefig(out, dpi=300)
    plt.close()
    print("Saved:", out)


In [ ]:
from collections import Counter
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

region_pair_counts = defaultdict(lambda: defaultdict(int))
region_pair_weights = defaultdict(lambda: defaultdict(list))

for stim_label, (avg_strength, stability, n_trials) in stim_edge_data.items():
    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    region_labels = unit_labels_permuted

    for i in range(len(units)):
        for j in range(len(units)):
            weight = avg_strength_perm[i, j]
            if weight == 0:
                continue
            source_region = region_labels[i]
            target_region = region_labels[j]
            region_key = (source_region, target_region)
            region_pair_counts[stim_label][region_key] += 1
            region_pair_weights[stim_label][region_key].append(weight)

# ---------- 1. BAR PLOT (MATCHING GRAPH): Number of stable neuron connections between brain regions ----------

bar_data = []

for stim_label in stim_label_list:
    avg_strength, stability, _ = stim_edge_data[stim_label]
    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]
    region_labels = unit_labels_permuted

    for i in range(len(units)):
        for j in range(len(units)):
            weight = avg_strength_perm[i, j]
            stable = stability_perm[i, j]
            if weight == 0 or stable < 0.3:
                continue
            source_region = region_labels[i]
            target_region = region_labels[j]
            bar_data.append({
                'Stimulus': stim_label,
                'Connection': f"{source_region} → {target_region}",
                'Count': 1  # Each valid edge contributes one count
            })

# Summarize counts
bar_df = pd.DataFrame(bar_data)
bar_df = bar_df.groupby(['Stimulus', 'Connection']).sum().reset_index()

plt.figure(figsize=(14, 6))
sns.barplot(data=bar_df, x='Connection', y='Count', hue='Stimulus')
plt.xticks(rotation=90)
plt.title("Number of Stable Neuron Connections Between Brain Regions (After Averaging & Stability Filter)")
plt.tight_layout()
plt.savefig(f"{plot_save_directory}barplot_region_connections_stable.png", dpi=300)
plt.close()
print("📊 Saved: barplot_region_connections_stable.png")


# ---------- 2. BOX PLOT: Edge weights for strong region pairs ----------
box_data = []
cutoff_fraction = 0.10  # 10% of possible connections between a given region pair

# Count neurons per region
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(unit_labels_permuted):
    region_neuron_indices[label].append(idx)

# Identify eligible region pairs per stimulus based on per-region connection density
eligible_pairs = set()

for stim_label in stim_label_list:
    for (source, target), count in region_pair_counts[stim_label].items():
        possible_edges = len(region_neuron_indices[source]) * len(region_neuron_indices[target])
        if possible_edges == 0:
            continue
        density = count / possible_edges
        if density > cutoff_fraction:
            eligible_pairs.add((source, target))

# Gather edge weights only for those region pairs
for stim_label in stim_label_list:
    for (source, target) in eligible_pairs:
        weights = region_pair_weights[stim_label].get((source, target), [])
        for w in weights:
            box_data.append({
                'Stimulus': stim_label,
                'Region Pair': f"{source} → {target}",
                'Edge Weight': w
            })

box_df = pd.DataFrame(box_data)

if not box_df.empty:
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=box_df, x='Region Pair', y='Edge Weight', hue='Stimulus')
    plt.xticks(rotation=90)
    #plt.title("Edge Weight Distributions (10%+ Region Pair Density)")
    plt.title("Edge Weights for Region Pairs with ≥10% of Possible Connections")
    plt.tight_layout()
    plt.savefig(f"{plot_save_directory}boxplot_edge_weights_density_filtered.png", dpi=300)
    plt.close()
    print("📦 Saved: boxplot_edge_weights_density_filtered.png")
else:
    print("⚠️ No region pairs passed the 10% density threshold.")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.lines import Line2D

# --- BOOTSTRAP PARAMETERS ---
n_boot = 50       # number of bootstrap resamples
min_stab = 0.8    # your 80% stability threshold
save_dir = save_directory
fig_dir  = plot_save_directory
os.makedirs(fig_dir, exist_ok=True)

# pre‐compute your denominator per source region
denoms = {src: len(region_neuron_indices[src])**2 
          for src in filtered_region_labels}

# we'll collect bootstrap %‐values per (stim, source_region, conn)
boot_vals = defaultdict(lambda: [])

for b in range(n_boot):
    # for each stimulus, sample with replacement from its idx files
    for stim in stim_label_list:
        ls = 3 if stim=='flashes' else 60
        # draw ls indices WITH replacement
        draw = np.random.choice(ls, size=ls, replace=True)
        
        # zero‐out presence accumulator
        edge_presence = np.zeros((len(units), len(units)))
        unit2full = {u:i for i,u in enumerate(units)}
        
        for idx in draw:
            fn = f"{save_dir}/ID{sess_id}_{stim}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
            if not os.path.exists(fn):
                continue
            with open(fn,'rb') as f:
                res = pkl.load(f)
            ce = res['ce']
            ce = ce - np.diag(np.diag(ce))
            trial_units = res.get('units', units_to_use_idx_[stim])
            try:
                idx_map = [unit2full[u] for u in trial_units]
            except KeyError:
                continue
            for i,ii in enumerate(idx_map):
                for j,jj in enumerate(idx_map):
                    if ce[i,j] != 0:
                        edge_presence[ii,jj] += 1
        
        # compute stability matrix under this bootstrap draw
        stability = edge_presence / ls
        stability[stability < min_stab] = 0
        
        # now for every source_region→conn, count pairs
        for source in filtered_region_labels:
            src_idxs = [i for i,lab in enumerate(unit_labels_permuted) if lab==source]
            denom = denoms[source]
            for conn in sorted(set(grouped['Connection'])):
                # skip connections not from this source
                tgt = conn.split(' → ')[1]
                if broad_region_map[source] != broad_region_map[conn.split(' → ')[0]]:
                    # only count same broad_source group
                    pass
                # collect all (i,j) pairs for this region‐→conn
                tgt_idxs = [j for j,lab in enumerate(unit_labels_permuted) if lab==tgt]
                # count how many survived threshold
                cnt = 0
                for i in src_idxs:
                    cnt += np.count_nonzero(stability[i, tgt_idxs])
                pct = cnt / denom * 100
                boot_vals[(stim, source, conn)].append(pct)

# Now your original bar heights are simply the mean over the BOOTSTRAP samples:
# and the SEM is the std/√n_boot

bar_w = 0.2
gap   = 1.0

for broad in sorted(set(broad_region_map[r] for r in filtered_region_labels)):
    # find all connections in this broad source
    conns = sorted({conn 
                    for (stim,src,conn),arr in boot_vals.items()
                    if broad_region_map[src]==broad})
    if not conns:
        continue

    xs, ys, es, cs, labs = [], [], [], [], []
    for i,conn in enumerate(conns):
        for j,stim in enumerate(ordered_stimuli):
            key = (stim, conn.split(' → ')[0], conn)
            if key not in boot_vals:
                continue
            arr = np.array(boot_vals[key])
            mean_pct = arr.mean()
            sem_pct  = arr.std(ddof=1)/np.sqrt(len(arr))

            x = i*gap + (j - len(ordered_stimuli)/2)*bar_w + bar_w/2
            xs.append(x); ys.append(mean_pct); es.append(sem_pct)
            cs.append(stim_color_map[stim]); labs.append(conn)

    # clean up x‐labels
    clean = []; prev = None
    for L in labs:
        clean.append(L if L!=prev else "")
        prev = L

    plt.figure(figsize=(max(10,len(xs)*0.3),6))
    plt.bar(xs, ys, yerr=es, capsize=4, color=cs, width=bar_w)
    plt.xticks(xs, clean, rotation=90)
    plt.ylabel('% of Possible Neuron Pairs')
    n_neurons = sum(len(region_neuron_indices[r]) 
                    for r in filtered_region_labels 
                    if broad_region_map[r]==broad)
    plt.title(f"{broad} (n={n_neurons} neurons): % Stable Connections ≥ {int(min_stab*100)}%")
    handles = [Line2D([0],[0],color=stim_color_map[s],lw=6,label=s)
               for s in ordered_stimuli]
    plt.legend(handles=handles, title='Stimulus', loc='upper right')
    plt.tight_layout()
    out = os.path.join(fig_dir, f"boot_bar_{broad.replace(' ','_')}.png")
    plt.savefig(out, dpi=300)
    plt.close()
    print("Saved:",out)


In [ ]:
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = 'save/'
plot_save_directory = 'figs/'
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.5  # NEW: Stability threshold used everywhere

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- GRAPHS ---
for stim_label in stim_label_list:
    for matrix_type in ['avg_strength', 'avg_strength_nonzero']:
        avg_strength = stim_edge_data[stim_label][matrix_type]
        stability = stim_edge_data[stim_label]['stability']

        G = nx.DiGraph()
        for i in range(len(units)):
            label = unit_labels_permuted[i]
            G.add_node(i, area=label)

        for i in range(len(units)):
            for j in range(len(units)):
                w = avg_strength[i, j]
                s = stability[i, j]
                if w != 0:
                    G.add_edge(i, j, weight=w, stability=s)

        pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
        fig, ax = plt.subplots(figsize=(12, 10))
        node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]

        weights = [np.abs(d['weight']) for _, _, d in G.edges(data=True)]
        stabilities = [d['stability'] for _, _, d in G.edges(data=True)]
        min_alpha = 0.3
        max_width_scale = 1.0

        for (u, v, d) in G.edges(data=True):
            alpha_val = min_alpha + (1 - min_alpha) * d['stability']
            width_val = np.interp(np.abs(d['weight']), [min(weights), max(weights)], [0.1, max_width_scale])
            nx.draw_networkx_edges(
                G, pos,
                edgelist=[(u, v)],
                width=width_val,
                alpha=alpha_val,
                edge_color='gray',
                arrows=True,
                ax=ax,
                connectionstyle='arc3,rad=0.2'
            )

        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
        nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

        subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
        legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
        ax.legend(handles=legend_patches, title='Brain Subregions', loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8, frameon=False)
        fig.subplots_adjust(bottom=0.30, right=0.72)

        ax.set_title(f'{stim_label} – {matrix_type.replace("_", " ").title()}')
        ax.axis('off')
        fig.savefig(f"{plot_save_directory}/graph_{stim_label}_{matrix_type}.png", dpi=300, bbox_inches='tight')
        plt.close()


In [ ]:
# --- FULL SCRIPT CONTINUED ---
# --- BAR PLOTS FOR BOTH MATRIX TYPES ---
region_labels = unit_labels_permuted
region_neuron_indices = defaultdict(list)
for idx, label in enumerate(region_labels):
    region_neuron_indices[label].append(idx)

for matrix_type in ['avg_strength', 'avg_strength_nonzero']:
    bar_data = []
    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][matrix_type]
        for i in range(len(units)):
            for j in range(len(units)):
                weight = avg_strength[i, j]
                if weight == 0:
                    continue
                source = region_labels[i]
                target = region_labels[j]
                bar_data.append({'Stimulus': stim_label, 'Connection': f"{source} → {target}", 'Count': 1})

    bar_df = pd.DataFrame(bar_data)
    bar_df = bar_df.groupby(['Stimulus', 'Connection']).sum().reset_index()

    plt.figure(figsize=(14, 6))
    sns.barplot(data=bar_df, x='Connection', y='Count', hue='Stimulus')
    plt.xticks(rotation=90)
    plt.title(f"Number of Stable Connections – {matrix_type.replace('_', ' ').title()}")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_directory, f"barplot_region_connections_{matrix_type}.png"), dpi=300)
    plt.close()

# --- BOX PLOTS FOR DENSE REGION PAIRS ---
cutoff_fraction = 0.10
for matrix_type in ['avg_strength', 'avg_strength_nonzero']:
    region_pair_counts = defaultdict(lambda: defaultdict(int))
    region_pair_weights = defaultdict(lambda: defaultdict(list))

    for stim_label in stim_label_list:
        avg_strength = stim_edge_data[stim_label][matrix_type]
        for i in range(len(units)):
            for j in range(len(units)):
                w = avg_strength[i, j]
                if w == 0:
                    continue
                source = region_labels[i]
                target = region_labels[j]
                region_pair_counts[stim_label][(source, target)] += 1
                region_pair_weights[stim_label][(source, target)].append(w)

    eligible_pairs = set()
    for stim_label in stim_label_list:
        for (source, target), count in region_pair_counts[stim_label].items():
            possible = len(region_neuron_indices[source]) * len(region_neuron_indices[target])
            if possible == 0:
                continue
            if count / possible >= cutoff_fraction:
                eligible_pairs.add((source, target))

    box_data = []
    for stim_label in stim_label_list:
        for (source, target) in eligible_pairs:
            for w in region_pair_weights[stim_label].get((source, target), []):
                box_data.append({
                    'Stimulus': stim_label,
                    'Region Pair': f"{source} → {target}",
                    'Edge Weight': w
                })

    box_df = pd.DataFrame(box_data)
    if not box_df.empty:
        plt.figure(figsize=(14, 6))
        sns.boxplot(data=box_df, x='Region Pair', y='Edge Weight', hue='Stimulus')
        plt.xticks(rotation=90)
        plt.title(f"Edge Weights (Dense Region Pairs) – {matrix_type.replace('_', ' ').title()}")
        plt.tight_layout()
        plt.savefig(os.path.join(plot_save_directory, f"boxplot_edge_weights_{matrix_type}.png"), dpi=300)
        plt.close()

In [ ]:
#%%tsci-plots
from neuropixels_testresults import convert_fcmat2units
import os
import json
import shutil
import time
import numpy as np
import pandas as pd
from scipy.special.orthogonal import chebyc
from tqdm import tqdm
import pickle as pkl
import matplotlib
import matplotlib.pyplot as plt
#from smoothspikes import *
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
#from sklearn.covariance import GraphicalLassoCV
#%%
data_directory = 'data/' # must be updated to a valid directory in your filesystem

#%%
sess_id = 791319847
#%%
data = cache.get_session_data(sess_id)
#%%
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)

#save_directory = 'D:\\OneDrive - UW\\research\\savedir'
bin_size=0.01
kernel_width=0.1

stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
method_name = 'tsci'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'save/'#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.05#0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix(convert_fcmat2units(ce-np.diag(np.diag(ce)), stim_label, units,permute),0.05,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)
#%%parcorr-plots

In [ ]:
units_to_use_idx_

In [ ]:

method_name = 'parcorr'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix_bw(convert_fcmat2units(ce, stim_label, units,permute),0.25,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)

#%%gc plots
method_name = 'gc'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.01,str(idx),allencon_save,'FMalpha'+str(alpha),'gc')
        #out[method_name] = np.asarray(out[method_name])
#%%gc3 plots
method_name = 'gc3'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        #ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce#ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.05,str(idx),allencon_save,'FMalpha'+str(alpha),'gc3')
        #out[method_name] = np.asarray(out[method_name])
#%%mle plots
method_name = 'glasso'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#2
    elif idx0==3:
        idxlist=[1]#1
    else:
        idxlist=[14]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        ce = ce - 2*np.diag(np.diag(ce))
        #ce = ce - np.diag(np.diag(ce))
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
        #plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.2,str(idx),allencon_save,'FM',method_name)
        #out[method_name] = np.asarray(out[method_name])


In [ ]:
#%%neuro-pc plots
method_name = 'neuropc'
bin_size = 0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.3
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_thresh{}_idx{}_niter{}.p'.format(bin_size,alpha,thresh,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            #out[method_name] = np.asarray(out[method_name])
            #ce = ce - np.diag(np.diag(ce))
            plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.5,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)


In [ ]:
#%%tsci-plots
method_name = 'tsci'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.25,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)
#%%parcorr-plots
method_name = 'parcorr'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix_bw(convert_fcmat2units(ce, stim_label, units,permute),0.25,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)

#%%gc plots
method_name = 'gc'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.01,str(idx),allencon_save,'FMalpha'+str(alpha),'gc')
        #out[method_name] = np.asarray(out[method_name])
#%%gc3 plots
method_name = 'gc3'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.loDocumentsad(open(results_file,'rb'))
        ce = result['ce']
        #ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce#ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.05,str(idx),allencon_save,'FMalpha'+str(alpha),'gc3')
        #out[method_name] = np.asarray(out[method_name])
#%%mle plots
method_name = 'glasso'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#2
    elif idx0==3:
        idxlist=[1]#1
    else:
        idxlist=[14]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        ce = ce - 2*np.diag(np.diag(ce))
        #ce = ce - np.diag(np.diag(ce))
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
        #plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.2,str(idx),allencon_save,'FM',method_name)
        #out[method_name] = np.asarray(out[method_name])

In [ ]:
#%%#%%
import os
import json
import shutil
import time
import numpy as np
import pandas as pd
from scipy.special.orthogonal import chebyc
from tqdm import tqdm
import pickle as pkl
import matplotlib
import matplotlib.pyplot as plt
#from smoothspikes import *
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
#from sklearn.covariance import GraphicalLassoCV
#%%
data_directory = 'D:\\OneDrive - UW\\research\\data@aibs' # must be updated to a valid directory in your filesystem

manifest_path = os.path.join(data_directory, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
sessions = cache.get_session_table()
probes = cache.get_probes()
channels = cache.get_channels()
units = cache.get_units()
#%%
sess_id = 791319847
#%%
data = cache.get_session_data(sess_id)
#%%
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}\\ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)

#save_directory = 'D:\\OneDrive - UW\\research\\savedir'
data_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\data'
#%%
stim_label='drifting_gratings_contrast'
allencon = A_Config(data_directory,sess_id,stim_label)
#sess_id=794812542
stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
#stim_label_list = ['natural_scenes','static_gratings','drifting_gratings','gabors','flashes']
#stim_label_list = ['drifting_gratings_contrast','gabors','flashes']
bin_size=0.01
kernel_width=0.1
#kernel_param=kernel_width/bin_size
#%%
X={}
P={}
for i, stim_label in enumerate(stim_label_list):
    print(i)
    allencon = A_Config(data_directory,sess_id,stim_label)
    P[stim_label] = pkl.load(open(allencon.name+'_bin_{}_P.p'.format(bin_size),'rb'))
    # with open(allencon.name+'_bin_{}_P.json'.format(bin_size),'w') as f:
    #     json.dump(P[stim_label],f)
    # f.close()
    #X[stim_label] = smooth_spikes(P[stim_label].values,kernel_width,bin_size)
    # with open(allencon.name+'_bin_{}_X.json'.format(bin_size),'w') as f:
    #     json.dump(X[stim_label],f)
    # f.close()
#%%
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_ = np.asarray(np.mean(np.sum(P[stim_label]>0, axis=1),axis=0)>(2*P[stim_label].shape[1]/10))
    print(np.sum(units_to_use_idx_))
    pkl.dump(units_to_use_idx_,open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'wb'))

#%%
units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

#%%
for stim_label in stim_label_list[:-1]:
    for idx in range(60):
        allencon = A_Config(data_directory,sess_id,stim_label)
        mul = 50
        idx1 = mul*idx
        X_out = X[stim_label][idx1,:,:]
        for idx2 in range(idx1+1,idx1+mul):
            X_out = np.row_stack((X_out,X[stim_label][idx2,:,:]))
        pkl.dump(X_out,open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'wb'))
for stim_label in stim_label_list[3:]:
    for idx in range(3):
        mul=50
        idx1 = mul*idx
        allencon = A_Config(data_directory,sess_id,stim_label)
        X_out = X[stim_label][idx1,:,:]
        for idx2 in range(idx1+1,idx1+mul):
            X_out = np.row_stack((X_out,X[stim_label][idx2,:,:]))
        pkl.dump(X_out,open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'wb'))
#%%
stim_label=stim_label_list[0]
allencon = A_Config(data_directory,sess_id,stim_label)
X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
X_load.shape
#%% units labeled
units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]
#%%
def plot_matrix(mat,vmnmx,id,allencon,namedetails='',method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    #m = mat - np.diag(np.diag(mat))
    m = mat
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    # if method == 'neuropc':
    #     vmnmx = 1#3
    # elif method == 'glm':
    #     vmnmx = 0.0008#0.001
    # elif method == 'granger':
    #     vmnmx = 6
    norm = mpl.colors.Normalize(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))##
    img=ax.matshow(m,cmap='RdBu_r',norm=norm)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(allencon.name+id+'_'+method+'_'+namedetails+'.png',format='png',dpi=600)
def plot_matrix_bw(mat,vmnmx,id,allencon,namedetails='',method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    #m = mat - np.diag(np.diag(mat))
    m = mat
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    # if method == 'neuropc':
    #     vmnmx = 1#3
    # elif method == 'glm':
    #     vmnmx = 0.0008#0.001
    # elif method == 'granger':
    #     vmnmx = 6
    #norm = mpl.colors.Normalize(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))##
    cmap = mpl.colors.ListedColormap(['white', 'black'])
    img=ax.matshow(m!=0,cmap=cmap)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(allencon.name+id+'_bw_'+method+'_'+namedetails+'.png',format='png',dpi=600)

#%%neuro-pc plots
method_name = 'neuropc'
bin_size = 0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.3
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_thresh{}_idx{}_niter{}.p'.format(bin_size,alpha,thresh,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            #out[method_name] = np.asarray(out[method_name])
            #ce = ce - np.diag(np.diag(ce))
            plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.5,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)
#%%tsci-plots
method_name = 'tsci'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.25,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)
#%%parcorr-plots
method_name = 'parcorr'
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'
#for alpha in [0.2,0.4]:
#    for thresh in thresh_list:
#        for idx in range(100):
#            for stim_label in stim_label_list[:-1]:
ce_store={}
out={}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    alpha = 0.1
    for thresh in [0.01]:
        if idx0==0:
            idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
        elif idx0==2:
            idxlist=[2]#[0,2,3,4,5,6,7,8,9]#2
            if alpha == 0.3:
                idxlist=[3]
        elif idx0==3:
            idxlist=[1]#[0,1,2]#1
        else:
            idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
        ce_avg=0
        for idx in idxlist:
            allencon_save = A_Config(save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            ce = result['ce']
            adj = result['adj']
            #ce[adj==0]=0
            
            #ce[ce<(np.max(ce)/10)]=0
            ce[np.isnan(ce)]=0
            out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
            plot_matrix_bw(convert_fcmat2units(ce, stim_label, units,permute),0.25,str(idx),allencon_save,'FMthresh'+str(thresh)+'_alpha'+str(alpha),method_name)

#%%gc plots
method_name = 'gc'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.01,str(idx),allencon_save,'FMalpha'+str(alpha),'gc')
        #out[method_name] = np.asarray(out[method_name])
#%%gc3 plots
method_name = 'gc3'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out = {}
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#[0,2,3,4,5,6,7,8,9]#2
    elif idx0==3:
        idxlist=[1]#[0,1,2]#1
    else:
        idxlist=[14]#[10,11,12,13,14,15,16,17,18,19]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.loDocumentsad(open(results_file,'rb'))
        ce = result['ce']
        #ce2= result['ce2']
        adj = result['adj']
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        #ce2[adj==0]=0
        gcval = ce#ce2*np.sign(ce)
        out[method_name].append(convert_fcmat2units(gcval, stim_label,units,permute))
        plot_matrix(convert_fcmat2units(gcval, stim_label,units,permute),0.05,str(idx),allencon_save,'FMalpha'+str(alpha),'gc3')
        #out[method_name] = np.asarray(out[method_name])
#%%mle plots
method_name = 'glasso'
bin_size=0.01
niter = 50
thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01]
save_directory = 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name
# for alpha in [0.2]:
#     for idx in range(100):
#        for stim_label in stim_label_list[:-1]:
alpha = 0.1
out[method_name]=[]
for idx0 in range(4):
    stim_label=stim_label_list[idx0]
    if idx0==0:
        idxlist=[54]#54
    elif idx0==2:
        idxlist=[2]#idxlist=[2,3]#2
    elif idx0==3:
        idxlist=[1]#1
    else:
        idxlist=[14]#14
    for idx in idxlist:
        allencon_save = A_Config(save_directory,sess_id,stim_label)
        results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)
        result = pkl.load(open(results_file,'rb'))
        ce = result['ce']
        ce2= result['ce2']
        adj = result['adj']
        ce = ce - 2*np.diag(np.diag(ce))
        #ce = ce - np.diag(np.diag(ce))
        #ce[ce<(np.max(ce)/10)]=0
        #ce[np.isnan(ce)]=0
        out[method_name].append(convert_fcmat2units(ce, stim_label, units,permute))
        #plot_matrix(convert_fcmat2units(ce, stim_label, units,permute),0.2,str(idx),allencon_save,'FM',method_name)
        #out[method_name] = np.asarray(out[method_name])
#%%
np.corrcoef(out['neuropc'][0].flatten(),out['glasso'][0].flatten())
np.corrcoef(out['gc'][0].flatten(),out['glasso'][0].flatten())
#%%
mean_adj ={}
for alpha in [0.2,0.4]:
    mean_adj[alpha]={}
    for stim_label in stim_label_list[:-1]:
        res_mean = 0
        for idx in range(10):
            allencon_save = A_Config(data_directory,save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            res_mean += result['adj']
        res_mean=res_mean/10
        mean_adj[alpha][stim_label] = res_mean    
#%%
alpha=0.2
stim_label=stim_label_list[2]
plot_matrix(mean_adj[alpha][stim_label],'neuropc',str(idx),allencon_save)        
#%%
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for unit in range(1,11):
        for trial in range(1,11):
            fig, ax = plt.subplots(1,1,figsize=(10,6))
            #ax.plot(P[stim_label][trial,:,unit]/100,linewidth=3,color="orange")
            ax.plot(X[stim_label][trial,:,unit]/100,linewidth=3,color="blue")
            ax.tick_params(labelsize=20)
            ax.set_title(stim_label+', neuron '+ str(unit) + ', trial '+ str(trial),fontsize=20)
            fig.savefig(allencon.name+'_X_'+'unit_'+str(unit)+'_trial_'+str(trial))
#%%
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for alpha in [0.4]:#,0.3,0.4]:
        print(alpha)
        n_trial = X[stim_label].shape[0]
        n_time = X[stim_label].shape[1]
        n_unit = X[stim_label].shape[2]
        dither_noise = 0.1
        adj_mats = []
        unit_list = []
        graphs = []
        #for k in range(10,11):
            #print(k)
        for k in range(25):
            print(k)
            units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0)>7#np.std(X[stim_label][k,:,:],axis=0)>10
            model = GraphicalLassoCV()
            model.fit(X[stim_label][k,:,units_to_use_idx_].T)
            cov_ = model.covariance_
            prec_ = model.precision_
            #adj_mats = prec_
            # lag = 3
            # #alpha=0.1
            # motif = stim_label
            # niter=1
            # #%%
            #     #t_ = time.time()
            # G,c1,c2= pc_plot_realdata(X[stim_label][k,:,units_to_use_idx_].T,lag,alpha,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),niter)
            #     #tiktok = time.time()-t_
            adj_mats.append(prec_)
            unit_list.append(units_to_use_idx_)
            # #print('{}s'.format(time.time()-t_))
            # pkl.dump(G,open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c1,open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c2,open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))

            # adj_mats.append(c2)
            # graphs.append(G)
            # unit_list.append(units_to_use_idx_)
        #pkl.dump((adj_mats, unit_list,graphs), open(allencon.name + '_bin{}_alpha_{}_adjmats_10.p'.format(bin_size,alpha), 'wb'))
        pkl.dump((adj_mats, unit_list), open(allencon.name + '_bin{}_adjmats_25.p'.format(bin_size,alpha), 'wb'))

            # allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
            # pkl.dump(G,open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c1,open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c2,open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
#%%neuropc
for stim_label in stim_label_list:
    print(stim_label)
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for alpha in [0.4]:
        adj_mats=[]
        unit_list=[]
        graphs=[]
        lag = 3
        niter=1
        for k in range(25):
            units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0)>7#np.std(X[stim_label][k,:,:],axis=0)>10
            graphs.append(pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb')))
            adj_mats.append(pkl.load(open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb')))
            unit_list.append(units_to_use_idx_)
        pkl.dump((adj_mats, unit_list,graphs), open(allencon.name + '_bin{}_alpha_{}_adjmats_25.p'.format(bin_size,alpha), 'wb'))

#%%
n_trials = len(adj_mats)
adj_x = np.zeros((n_trials, n_unit*n_unit))
for alpha in [0.1,0.2,0.3,0.4]:
    for trial, mat in enumerate(adj_mats):
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat(nz_idcs, l_nz))), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        adj_x[trial, :] = full_adj.flatten()
    pkl.dump(adj_x, open(allencon.name + '_bin{}_alpha_{}_adjmats_10_allvecs.p'.format(bin_size,alpha), 'wb'))

#%%neuropc
c1_cluster = {}
for idx in range(4):
    stim_label=stim_label_list[idx]
    lag=3
    niter=1
    k=4
    #units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    #units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
    alpha=0.4
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c1=pkl.load(open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c1_cluster[idx]=c1
#%%
for idx in range(4):
    c1=c1_cluster[idx]
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    plot_matrix(c1,'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff',allencon,method='neuropc')
#plot_graph(G,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'graph',allencon)
#%%
stim_label=stim_label_list[3]
for k in range(30):
    lag=3
    niter=1
    #units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    #units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
    alpha=0.4
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c2=pkl.load(open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    plot_matrix(c2,'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff',allencon,method='neuropc')

#%%
import pickle as pkl
import numpy as np
motif = 'motif_large'
m=20
bin_size=0.01
lag=6
niter=50
#units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
#units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
alpha=0.4
save_directory = 'D:\\OneDrive - UW\\research\\savedir\\'
m1=4
a=np.zeros((m1,m1))
a[0,2]=a[1,2]=a[2,3]=0.5
n=5
m=m1*n
w=np.kron(np.eye(n,dtype=int),a)
for i in range(n-1):
    w[i*4 + 3] = 0.5
#allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
#G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
c2=pkl.load(open(save_directory+motif+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_m_{}.p'.format(bin_size,alpha,lag,niter,m),'rb'))
plot_matrix_simul(c2,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_m_'+str(m),method='neuropc')

#plot_matrix_simul(c2,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_m_'+str(m),method='neuropc')
#%%
i=0
mat=[]
#%%
stim_label = stim_label_list[0]
method='granger'
save_directory = 'D:\\OneDrive - UW\\research\\savedir\\'
file='ID791319847_natural_scenes_bin_0.01_gc_trial_0_idx_'+str(i)#'ID791319847_natural_scenes_bin_0.01_c2_alpha_0.4_lag_1_niter_2_trial_4_iter_'+str(i)+'val'#
c2 = pkl.load(open(save_directory+file+'.p','rb'))
allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
plot_matrix(c2,file,'causaleff',allencon,method)
mat.append(c2)
print(i)
i=i+1
#%%for glm
for idx in range(4):
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir\\glm'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    from sklearn.covariance import GraphicalLassoCV
    n_trial = X[stim_label].shape[0]
    n_time = X[stim_label].shape[1]
    n_unit = X[stim_label].shape[2]
    units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    model = GraphicalLassoCV()
    model.fit(X[stim_label][k,:,units_to_use_idx_].T)
    cov_ = model.covariance_
    prec_ = model.precision_
    adj_mat = prec_

    #adj_mat, unit_list=pkl.load(open(allencon.name + '_bin{}_adjmats_10_glm.p'.format(bin_size), 'rb'))
    plot_matrix(adj_mat,'_trial_'+str(k),'glm',allencon,method='glm')

    #plot_matrix(np.multiply(adj_mat,c2),'_trial_'+str(k),'glmXneuropc',allencon)
#%%granger
for idx in range(3,4):
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    G=pkl.load(open(allencon.name+'_bin_{}_gc_trial_{}.p'.format(bin_size,k),'rb'))
    plot_matrix(G,'_trial_'+str(k),'grangercausal',allencon,method='granger')
#plot_graph
#%%
units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8

#%%individual plotting adj_mat and graph
for idx in range(3):
    stim_label=stim_label_list[idx]
    lag=3
    niter=1
    for k in range(10):
        #k=4
        allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
        for alpha in [0.2,0.3,0.4]:
            #alpha=0.4
            G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
            c2=pkl.load(open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
            plot_matrix(c2,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff')
            plot_graph(G,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'graph')
#%%
plot_matrix(c1, motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'pc_causaleff')
plot_matrix(c2, motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'pc_causaleff')
nx.draw(G, with_labels= True,font_color='white',pos=nx.circular_layout(G))
plot_graph(G,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'graph')
# %%

#%%
def plot_matrix_simul(mat,motif,method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    m = mat - np.diag(np.diag(mat))
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    #if method == 'neuropc':
    #    vmnmx = 3
    #elif method == 'glm':
    #    vmnmx = 0.001
    #elif method == 'granger':
    #    vmnmx = 5 
    norm = mpl.colors.Normalize(vmin=-np.max(m),vmax=np.max(m))#(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))
    img=ax.matshow(m,cmap='RdBu_r',norm=norm)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    ax.grid(axis='y')
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(str(motif)+'_'+str(method)+'.png',format='png',dpi=600)

#%%
def plot_graph(G,motif,id,allencon):
    import matplotlib.pyplot as plt
    import networkx as nx
    fig,ax = plt.subplots(1,1,figsize=(20,20))
    nx.draw(G, with_labels= True, font_color= 'white', pos= nx.circular_layout(G),ax=ax,font_size=20,node_size=1000)
    fig.savefig(allencon.name+id+'_'+str(motif)+'.png',format = 'png',dpi=600)
# %%
def smooth_spikes( P, kernel_width , bin_size ):
    # Rahul
    """
    P: #trials x #units x #time-points sized ndarray
    kernel_param: width of kernel (in samplingrate*1000 units)

    """

    import time
    import scipy.stats, scipy.signal
    import scipy as sp

    start = time.time() # tic Measures function running speed. Works only on Macs

    #trials = P.shape[0]
    #units = P.shape[1]
    #bins = P.shape[2]

    # Create fake data for testing purposes
    #    array = np.random.randint(0,2,(trials,bins))
    #    sigma = 45

    #Define kernel
    kernel_param = kernel_width/bin_size
    sigma = kernel_param/6.
    edges = np.arange(-3*sigma,3*sigma+1,1)
    kernel = sp.stats.norm.pdf(edges,0, sigma) #Use a gaussian function
    kernel = kernel/sum(kernel)
    
    #Compute Spike Density Function for all trials
    #X = np.zeros((trials,units,bins))
    X = sp.signal.convolve(P, kernel[None,:,None], mode='same',method='direct')

    print('Run time for SDF function was ' + str(round(time.time()-start)) + 'seconds') # toc
    return X

# %%
import numpy as np
import pandas as pd
import pickle as pkl

def save_Ps(cache,sess_id,stim_label_list,bin_size):
    data = cache.get_session_data(sess_id)
    #t_bin = {}
    #P = {}
    for i, stim_label in enumerate(stim_label_list):
        allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
        print(i)
        # get start, end time points
        stim_data_table = data.get_stimulus_table()[data.get_stimulus_table().stimulus_name==stim_label]
        start_time = np.array(stim_data_table.start_time)
        stop_time = np.array(stim_data_table.stop_time)
        trial_time = np.mean(stop_time - start_time)
        bin_edges = np.arange(trial_time + bin_size, step = bin_size )
        t_bin = bin_edges
        print(i)
        stim_presentation_ids = data.stimulus_presentations.loc[
            (data.stimulus_presentations['stimulus_name'] == stim_label)
            ].index.values
        print(i)
        unit_ids = data.units.index.values
        print(i)
        P = data.presentationwise_spike_counts(bin_edges,stim_presentation_ids,unit_ids)/bin_size
        pkl.dump(P,open(allencon.name+'_bin_{}_P.p'.format(bin_size),'wb'))
        pkl.dump(t_bin,open(allencon.name+'_bin_{}_t.p'.format(bin_size),'wb'))

#%%

# %%
save_Ps(cache, sess_id, stim_label_list = stim_label_list, bin_size=bin_size)
# %%neuropc
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list, graph = pkl.load(open(allencon.name + '_bin{}_alpha_{}_adjmats_25.p'.format(bin_size,alpha), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'wb'))

#%%
alpha=0.4
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list = pkl.load(open(allencon.name + '_bin{}_adjmats_25.p'.format(bin_size), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'wb'))

# %%granger
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list = pkl.load(open(allencon.name + '_bin{}_granger_adjmats_25.p'.format(bin_size), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'wb'))

# %%neuropc-hist
from scipy.stats import ttest_ind
alpha = 0.4

metric_1 = {}
pvals1 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'
 
for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals1[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals1[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals1[stim_label1+'_'+stim_label2]))):
        pvals1[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_1[stim_label1+'_'+stim_label2]=sum(pvals1[stim_label1+'_'+stim_label2]<=0.3)/len(pvals1[stim_label1+'_'+stim_label2])

    fig, ax = plt.subplots(1,1,figsize=(7,5))
    fnt=22
    ax.hist(pvals1[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.set_ylim(0,10**3)
    ax.set_xlim(0,1)
    ax.tick_params(labelsize=fnt)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=fnt)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'_alpha_'+str(alpha)+'.png',format='png',dpi=600)
#%%glm
from scipy.stats import ttest_ind
alpha = 0.4
ind1=0
ind2=1
metric_2 = {}
pvals2 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'

for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals2[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals2[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals2[stim_label1+'_'+stim_label2]))):
        pvals2[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_2[stim_label1+'_'+stim_label2]=sum(pvals2[stim_label1+'_'+stim_label2]<=0.3)/len(pvals2[stim_label1+'_'+stim_label2])
    fig, ax = plt.subplots(1,1,figsize=(7,5))
    fnt=22
    ax.hist(pvals2[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.set_ylim(0,15**3)
    ax.tick_params(labelsize=fnt)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=22)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'.png',format='png',dpi=600)
#%%granger
from scipy.stats import ttest_ind
alpha = 0.4

metric_1 = {}
pvals1 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'
 
for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals1[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals1[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals1[stim_label1+'_'+stim_label2]))):
        pvals1[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_1[stim_label1+'_'+stim_label2]=sum(pvals1[stim_label1+'_'+stim_label2]<=0.3)/len(pvals1[stim_label1+'_'+stim_label2])
    fnt=22
    fig, ax = plt.subplots(1,1,figsize=(7,5))
    ax.hist(pvals1[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.tick_params(labelsize=fnt)
    ax.set_ylim(0,10**4)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=fnt)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'_gc_'+str(alpha)+'.png',format='png',dpi=600)
#%%
fig, ax = plt.subplots(1,1,figsize=(7,5))
ax.hist(pvals)#,log=True)#log scale
ax.axvline(0.1,color="red")
ax.set_xlabel('p-values')
ax.set_title(stim_label1+' vs '+stim_label2)
allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
fig.savefig(allencon.name+'_alpha_'+str(alpha),format='png',dpi=600)
# %%
np.corrcoef(out['neuropc'][0].flatten(),out['glasso'][0].flatten())
np.corrcoef(out['gc'][0].flatten(),out['glasso'][0].flatten())
#%%
mean_adj ={}
for alpha in [0.2,0.4]:
    mean_adj[alpha]={}
    for stim_label in stim_label_list[:-1]:
        res_mean = 0
        for idx in range(10):
            allencon_save = A_Config(data_directory,save_directory,sess_id,stim_label)
            result = pkl.load(open(allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter),'rb'))
            res_mean += result['adj']
        res_mean=res_mean/10
        mean_adj[alpha][stim_label] = res_mean    
#%%
alpha=0.2
stim_label=stim_label_list[2]
plot_matrix(mean_adj[alpha][stim_label],'neuropc',str(idx),allencon_save)        
#%%
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for unit in range(1,11):
        for trial in range(1,11):
            fig, ax = plt.subplots(1,1,figsize=(10,6))
            #ax.plot(P[stim_label][trial,:,unit]/100,linewidth=3,color="orange")
            ax.plot(X[stim_label][trial,:,unit]/100,linewidth=3,color="blue")
            ax.tick_params(labelsize=20)
            ax.set_title(stim_label+', neuron '+ str(unit) + ', trial '+ str(trial),fontsize=20)
            fig.savefig(allencon.name+'_X_'+'unit_'+str(unit)+'_trial_'+str(trial))
#%%
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for alpha in [0.4]:#,0.3,0.4]:
        print(alpha)
        n_trial = X[stim_label].shape[0]
        n_time = X[stim_label].shape[1]
        n_unit = X[stim_label].shape[2]
        dither_noise = 0.1
        adj_mats = []
        unit_list = []
        graphs = []
        #for k in range(10,11):
            #print(k)
        for k in range(25):
            print(k)
            units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0)>7#np.std(X[stim_label][k,:,:],axis=0)>10
            model = GraphicalLassoCV()
            model.fit(X[stim_label][k,:,units_to_use_idx_].T)
            cov_ = model.covariance_
            prec_ = model.precision_
            #adj_mats = prec_
            # lag = 3
            # #alpha=0.1
            # motif = stim_label
            # niter=1
            # #%%
            #     #t_ = time.time()
            # G,c1,c2= pc_plot_realdata(X[stim_label][k,:,units_to_use_idx_].T,lag,alpha,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),niter)
            #     #tiktok = time.time()-t_
            adj_mats.append(prec_)
            unit_list.append(units_to_use_idx_)
            # #print('{}s'.format(time.time()-t_))
            # pkl.dump(G,open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c1,open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c2,open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))

            # adj_mats.append(c2)
            # graphs.append(G)
            # unit_list.append(units_to_use_idx_)
        #pkl.dump((adj_mats, unit_list,graphs), open(allencon.name + '_bin{}_alpha_{}_adjmats_10.p'.format(bin_size,alpha), 'wb'))
        pkl.dump((adj_mats, unit_list), open(allencon.name + '_bin{}_adjmats_25.p'.format(bin_size,alpha), 'wb'))

            # allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
            # pkl.dump(G,open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c1,open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
            # pkl.dump(c2,open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'wb'))
#%%neuropc
for stim_label in stim_label_list:
    print(stim_label)
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    for alpha in [0.4]:
        adj_mats=[]
        unit_list=[]
        graphs=[]
        lag = 3
        niter=1
        for k in range(25):
            units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0)>7#np.std(X[stim_label][k,:,:],axis=0)>10
            graphs.append(pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb')))
            adj_mats.append(pkl.load(open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb')))
            unit_list.append(units_to_use_idx_)
        pkl.dump((adj_mats, unit_list,graphs), open(allencon.name + '_bin{}_alpha_{}_adjmats_25.p'.format(bin_size,alpha), 'wb'))

#%%
n_trials = len(adj_mats)
adj_x = np.zeros((n_trials, n_unit*n_unit))
for alpha in [0.1,0.2,0.3,0.4]:
    for trial, mat in enumerate(adj_mats):
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat(nz_idcs, l_nz))), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        adj_x[trial, :] = full_adj.flatten()
    pkl.dump(adj_x, open(allencon.name + '_bin{}_alpha_{}_adjmats_10_allvecs.p'.format(bin_size,alpha), 'wb'))

#%%neuropc
c1_cluster = {}
for idx in range(4):
    stim_label=stim_label_list[idx]
    lag=3
    niter=1
    k=4
    #units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    #units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
    alpha=0.4
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c1=pkl.load(open(allencon.name+'_bin_{}_c1_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c1_cluster[idx]=c1
#%%
for idx in range(4):
    c1=c1_cluster[idx]
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    plot_matrix(c1,'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff',allencon,method='neuropc')
#plot_graph(G,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'graph',allencon)
#%%
stim_label=stim_label_list[3]
for k in range(30):
    lag=3
    niter=1
    #units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    #units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
    alpha=0.4
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    c2=pkl.load(open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    plot_matrix(c2,'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff',allencon,method='neuropc')

#%%
import pickle as pkl
import numpy as np
motif = 'motif_large'
m=20
bin_size=0.01
lag=6
niter=50
#units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
#units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8
alpha=0.4
save_directory = 'D:\\OneDrive - UW\\research\\savedir\\'
m1=4
a=np.zeros((m1,m1))
a[0,2]=a[1,2]=a[2,3]=0.5
n=5
m=m1*n
w=np.kron(np.eye(n,dtype=int),a)
for i in range(n-1):
    w[i*4 + 3] = 0.5
#allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
#G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
c2=pkl.load(open(save_directory+motif+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_m_{}.p'.format(bin_size,alpha,lag,niter,m),'rb'))
plot_matrix_simul(c2,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_m_'+str(m),method='neuropc')

#plot_matrix_simul(c2,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_m_'+str(m),method='neuropc')
#%%
i=0
mat=[]
#%%
stim_label = stim_label_list[0]
method='granger'
save_directory = 'D:\\OneDrive - UW\\research\\savedir\\'
file='ID791319847_natural_scenes_bin_0.01_gc_trial_0_idx_'+str(i)#'ID791319847_natural_scenes_bin_0.01_c2_alpha_0.4_lag_1_niter_2_trial_4_iter_'+str(i)+'val'#
c2 = pkl.load(open(save_directory+file+'.p','rb'))
allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
plot_matrix(c2,file,'causaleff',allencon,method)
mat.append(c2)
print(i)
i=i+1
#%%for glm
for idx in range(4):
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir\\glm'
    allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
    from sklearn.covariance import GraphicalLassoCV
    n_trial = X[stim_label].shape[0]
    n_time = X[stim_label].shape[1]
    n_unit = X[stim_label].shape[2]
    units_to_use_idx_ = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
    model = GraphicalLassoCV()
    model.fit(X[stim_label][k,:,units_to_use_idx_].T)
    cov_ = model.covariance_
    prec_ = model.precision_
    adj_mat = prec_

    #adj_mat, unit_list=pkl.load(open(allencon.name + '_bin{}_adjmats_10_glm.p'.format(bin_size), 'rb'))
    plot_matrix(adj_mat,'_trial_'+str(k),'glm',allencon,method='glm')

    #plot_matrix(np.multiply(adj_mat,c2),'_trial_'+str(k),'glmXneuropc',allencon)
#%%granger
for idx in range(3,4):
    stim_label=stim_label_list[idx]
    save_directory = 'D:\\OneDrive - UW\\research\\savedir'
    allencon = A_Config(save_directory,sess_id,stim_label)
    #G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
    G=pkl.load(open(allencon.name+'_bin_{}_gc_trial_{}.p'.format(bin_size,k),'rb'))
    plot_matrix(G,'_trial_'+str(k),'grangercausal',allencon,method='granger')
#plot_graph
#%%
units_to_use_idx_1 = np.sum(P[stim_label][k,:,:]>0,axis=0) > 7
units_to_use_idx_1_ = np.sum(P[stim_label][k,:,units_to_use_idx_1]>0,axis=0) > 8

#%%individual plotting adj_mat and graph
for idx in range(3):
    stim_label=stim_label_list[idx]
    lag=3
    niter=1
    for k in range(10):
        #k=4
        allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
        for alpha in [0.2,0.3,0.4]:
            #alpha=0.4
            G=pkl.load(open(allencon.name+'_bin_{}_G_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
            c2=pkl.load(open(allencon.name+'_bin_{}_c2_alpha_{}_lag_{}_niter_{}_trial_{}.p'.format(bin_size,alpha,lag,niter,k),'rb'))
            plot_matrix(c2,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'causaleff')
            plot_graph(G,str(sess_id)+'_'+stim_label+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter)+'_trial_'+str(k),'graph')
#%%
plot_matrix(c1, motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'pc_causaleff')
plot_matrix(c2, motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'pc_causaleff')
nx.draw(G, with_labels= True,font_color='white',pos=nx.circular_layout(G))
plot_graph(G,motif+'_alpha_'+str(alpha)+'_lag_'+str(lag)+'_niter_'+str(niter),'graph')
# %%

#%%
def plot_matrix_simul(mat,motif,method='neuropc'):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    fig,ax = plt.subplots(1,1,figsize=(14,14))
    m = mat - np.diag(np.diag(mat))
    #norm = mpl.colors.Normalize(vmin=0,vmax=np.max(m))
    #if method == 'neuropc':
    #    vmnmx = 3
    #elif method == 'glm':
    #    vmnmx = 0.001
    #elif method == 'granger':
    #    vmnmx = 5 
    norm = mpl.colors.Normalize(vmin=-np.max(m),vmax=np.max(m))#(vmin=-vmnmx,vmax=vmnmx)#(vmin=-np.max(m),vmax=np.max(m))
    img=ax.matshow(m,cmap='RdBu_r',norm=norm)
    fig.subplots_adjust(right=0.8)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    cbar_ax.tick_params(labelsize=20)
    fig.colorbar(img,cax=cbar_ax,extend = 'both')
    ax.set_ylabel("Neurons",fontsize=40)
    ax.set_xlabel("Neurons",fontsize=40)
    ax.grid(axis='y')
    #ax.tick_params(labelsize=60)
    #ax=set_xticks()
    #ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=30)
    #fig.savefig(id+'_'+str(motif)+'.pdf',format='pdf',dpi=600)
    fig.savefig(str(motif)+'_'+str(method)+'.png',format='png',dpi=600)

#%%
def plot_graph(G,motif,id,allencon):
    import matplotlib.pyplot as plt
    import networkx as nx
    fig,ax = plt.subplots(1,1,figsize=(20,20))
    nx.draw(G, with_labels= True, font_color= 'white', pos= nx.circular_layout(G),ax=ax,font_size=20,node_size=1000)
    fig.savefig(allencon.name+id+'_'+str(motif)+'.png',format = 'png',dpi=600)


# %%
import numpy as np
import pandas as pd
import pickle as pkl

def save_Ps(cache,sess_id,stim_label_list,bin_size):
    data = cache.get_session_data(sess_id)
    #t_bin = {}
    #P = {}
    for i, stim_label in enumerate(stim_label_list):
        allencon = A_Config(data_directory,save_directory,sess_id,stim_label)
        print(i)
        # get start, end time points
        stim_data_table = data.get_stimulus_table()[data.get_stimulus_table().stimulus_name==stim_label]
        start_time = np.array(stim_data_table.start_time)
        stop_time = np.array(stim_data_table.stop_time)
        trial_time = np.mean(stop_time - start_time)
        bin_edges = np.arange(trial_time + bin_size, step = bin_size )
        t_bin = bin_edges
        print(i)
        stim_presentation_ids = data.stimulus_presentations.loc[
            (data.stimulus_presentations['stimulus_name'] == stim_label)
            ].index.values
        print(i)
        unit_ids = data.units.index.values
        print(i)
        P = data.presentationwise_spike_counts(bin_edges,stim_presentation_ids,unit_ids)/bin_size
        pkl.dump(P,open(allencon.name+'_bin_{}_P.p'.format(bin_size),'wb'))
        pkl.dump(t_bin,open(allencon.name+'_bin_{}_t.p'.format(bin_size),'wb'))

#%%

# %%
save_Ps(cache, sess_id, stim_label_list = stim_label_list, bin_size=bin_size)
# %%neuropc
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list, graph = pkl.load(open(allencon.name + '_bin{}_alpha_{}_adjmats_25.p'.format(bin_size,alpha), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'wb'))

#%%
alpha=0.4
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list = pkl.load(open(allencon.name + '_bin{}_adjmats_25.p'.format(bin_size), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'wb'))

# %%granger
from scipy import sparse
for stim_label in stim_label_list:
    print(stim_label)
    #for alpha in [0.2,0.3,0.4]:
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label)
    adj_mats, unit_list = pkl.load(open(allencon.name + '_bin{}_granger_adjmats_25.p'.format(bin_size), 'rb'))
    n_trials = len(adj_mats)
    n_unit = X[stim_label].shape[2]
    n_time = X[stim_label].shape[1]
    #     print(n_unit*(n_unit+1)/2)
    adj_x = np.zeros((n_trials, int(n_unit*(n_unit-1)/2)))
    for trial, mat in enumerate(adj_mats):
        if sum(unit_list[trial]) != adj_mats[trial].shape[0]:
            unit_list[trial] = np.sum(P[stim_label][trial,:,:]>0,axis=0)>6
        a = unit_list[trial] * np.arange(1, len(unit_list[trial])+1)
        mask = np.ma.masked_equal(a,0)
        nz_idcs = mask.compressed() -1
        l_nz = len(nz_idcs)
        print(l_nz)
        print(adj_mats[trial].shape)        
        a_flat = adj_mats[trial].flatten()
        full_sparse = sparse.coo_matrix((a_flat, (np.repeat(nz_idcs, l_nz) , np.repeat([nz_idcs], l_nz, axis = 0).flatten())), shape=(n_unit, n_unit))
        full_adj = full_sparse.todense()
        iu_idcs = np.triu_indices(full_adj.shape[0],k=1)
        adj_x[trial, :] = full_adj[iu_idcs]
    pkl.dump(adj_x, open(allencon.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'wb'))

# %%neuropc-hist
from scipy.stats import ttest_ind
alpha = 0.4

metric_1 = {}
pvals1 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'
 
for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_alpha_{}_adjmats_25_allvecs.p'.format(bin_size,alpha), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals1[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals1[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals1[stim_label1+'_'+stim_label2]))):
        pvals1[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_1[stim_label1+'_'+stim_label2]=sum(pvals1[stim_label1+'_'+stim_label2]<=0.3)/len(pvals1[stim_label1+'_'+stim_label2])

    fig, ax = plt.subplots(1,1,figsize=(7,5))
    fnt=22
    ax.hist(pvals1[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.set_ylim(0,10**3)
    ax.set_xlim(0,1)
    ax.tick_params(labelsize=fnt)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=fnt)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'_alpha_'+str(alpha)+'.png',format='png',dpi=600)
#%%glm
from scipy.stats import ttest_ind
alpha = 0.4
ind1=0
ind2=1
metric_2 = {}
pvals2 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'

for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_adjmats_25_allvecs.p'.format(bin_size), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals2[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals2[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals2[stim_label1+'_'+stim_label2]))):
        pvals2[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_2[stim_label1+'_'+stim_label2]=sum(pvals2[stim_label1+'_'+stim_label2]<=0.3)/len(pvals2[stim_label1+'_'+stim_label2])
    fig, ax = plt.subplots(1,1,figsize=(7,5))
    fnt=22
    ax.hist(pvals2[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.set_ylim(0,15**3)
    ax.tick_params(labelsize=fnt)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=22)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'.png',format='png',dpi=600)
#%%granger
from scipy.stats import ttest_ind
alpha = 0.4

metric_1 = {}
pvals1 = {}
save_directory = 'D:\\OneDrive - UW\\research\\savedir'
 
for (ind1, ind2) in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    stim_label1=stim_label_list[ind1]
    allencon1 = A_Config(data_directory,save_directory,sess_id, stim_label1)
    stim_label2=stim_label_list[ind2]
    allencon2 = A_Config(data_directory,save_directory,sess_id, stim_label2)
    adj_mat1=pkl.load(open(allencon1.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'rb'))
    adj_mat2=pkl.load(open(allencon2.name + '_bin{}_gc_adjmats_25_allvecs.p'.format(bin_size), 'rb'))

    mask = np.logical_or(np.sum(adj_mat1,axis=0) !=0, np.sum(adj_mat2,axis=0) !=0)
    adj1masked = adj_mat1[:,mask]
    adj2masked = adj_mat2[:,mask]
    #pvals = np.zeros((adj_mat1.shape[1]))
    pvals1[stim_label1+'_'+stim_label2] = np.zeros((adj1masked.shape[1]))
    print(len(pvals1[stim_label1+'_'+stim_label2]))
   
    for i in tqdm(range(len(pvals1[stim_label1+'_'+stim_label2]))):
        pvals1[stim_label1+'_'+stim_label2][i]= ttest_ind(adj1masked[:,i],adj2masked[:,i])[1]
    metric_1[stim_label1+'_'+stim_label2]=sum(pvals1[stim_label1+'_'+stim_label2]<=0.3)/len(pvals1[stim_label1+'_'+stim_label2])
    fnt=22
    fig, ax = plt.subplots(1,1,figsize=(7,5))
    ax.hist(pvals1[stim_label1+'_'+stim_label2],log=True)#,log=True)#log scale
    ax.axvline(0.1,color="red")
    ax.tick_params(labelsize=fnt)
    ax.set_ylim(0,10**4)
    ax.set_xlabel('p-values',fontsize=fnt)
    ax.set_title(stim_label1+' vs '+stim_label2,fontsize=fnt)
    allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
    fig.savefig(allencon.name+'_gc_'+str(alpha)+'.png',format='png',dpi=600)
#%%
fig, ax = plt.subplots(1,1,figsize=(7,5))
ax.hist(pvals)#,log=True)#log scale
ax.axvline(0.1,color="red")
ax.set_xlabel('p-values')
ax.set_title(stim_label1+' vs '+stim_label2)
allencon = A_Config(data_directory,save_directory,sess_id, stim_label1+'_vs_'+stim_label2)
fig.savefig(allencon.name+'_alpha_'+str(alpha),format='png',dpi=600)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.spatial.distance import pdist, squareform
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

# === PARAMETERS ===
sess_id = 791319847
stim_label = "natural_scenes"
n_trials = 60
base_path = "save/"  # where your result .p files are stored
bin_size = 0.01
alpha = 0.05
niter = 50

# === LOAD UNIT METADATA AND DSI ===
cache = EcephysProjectCache.from_warehouse(manifest="data/manifest.json")
session = cache.get_session_data(sess_id)
unit_df = session.units

# Load units2use
with open(f"data/ID{sess_id}_{stim_label}_units2use_stim_{stim_label}.p", "rb") as f:
    units2use_bool = pickle.load(f)
unit_indices = np.where(units2use_bool)[0]
unit_out = unit_df.iloc[unit_indices, :]

# DSI vector
if "dsi_dg" in unit_out.columns:
    dsi = unit_out["dsi_dg"].values
elif "direction_selectivity_index" in unit_out.columns:
    dsi = unit_out["direction_selectivity_index"].values
else:
    raise ValueError("DSI field not found!")

dsi_dist = squareform(pdist(dsi[:, None], metric="euclidean"))

# === ACCUMULATE RESULTS ACROSS TRIALS ===
connected_means = []
nonconnected_means = []

for idx in range(n_trials):
    filename = f"{base_path}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
    if not os.path.exists(filename):
        print(f"Missing: {filename}")
        continue

    with open(filename, "rb") as f:
        result = pickle.load(f)
    ce = result["ce"]

    # Make sure diagonals are zeroed
    np.fill_diagonal(ce, 0)

    # Create masks
    connected_mask = ce != 0
    nonconnected_mask = ~connected_mask & ~np.eye(len(dsi), dtype=bool)

    # DSI distances
    conn_dsi = dsi_dist[connected_mask]
    nonconn_dsi = dsi_dist[nonconnected_mask]

    # Store means
    connected_means.append(np.mean(conn_dsi))
    nonconnected_means.append(np.mean(nonconn_dsi))

# === PLOT RESULTS ===
plt.figure(figsize=(8, 5))
plt.plot(connected_means, label="Connected", marker='o')
plt.plot(nonconnected_means, label="Not Connected", marker='x')
plt.xlabel("Trial Index")
plt.ylabel("Mean DSI Distance")
plt.title(f"DSI Distance Across Trials – {stim_label.replace('_', ' ').title()}")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
unit_out.columns